# K-suite — Colab T4 (K1–K7)

One runtime, **shared setup**, **separate downloadable CSVs** per arm.

| Order | Flag | Arm | Notes |
|------:|------|-----|-------|
| 1 | `RUN_K1_O5` | O5 teacher-forced likelihood | Highest priority. Smoke `LIMIT=2`, `DRY_RUN=True` first. Qwen primary; Llama skipped without `HF_TOKEN`. |
| 2 | `RUN_K2_O6` | O6 quantization sensitivity | After K1. Kill: median rank shift > 50 drops 4-bit ranks. |
| 3 | `RUN_K3_O15` | O15 surprisal contamination | First pass: `RUN_OPTIONAL_SCALE=False`. |
| 4 | `RUN_K4_O8` | O8 mech↔behavior link | Needs K1 CSV + O7 PASS (GSM). ALGO + GSM. |
| 5 | `RUN_K6_O14B` | O14b naming likelihood | Qwen 1.5B/3B; independent of O5 grid. |
| 6 | `RUN_K7_DS16` | DS-16 recognition/recall | Qwen 1.5B/3B; can+W3. |
| 7 | `RUN_K5_O16` | O16 open-model calibration | **Last.** Sparse corpus GT → expect uninformative AUC; still closes the loop. |

### Workflow
1. Set Colab secrets: `HF_TOKEN` (optional, Llama only), `GITHUB_TOKEN` if private clone.
2. Edit **master knobs** in the next code cell (`LIMIT`, `DRY_RUN`, `RUN_K*`).
3. **Runtime → Run all** (or run section-by-section).
4. After each arm (or at the end), use that arm’s **Download** cell — files stay distinct.

Land downloads in the repo per `colab/README.md` (`results/raw/` / `results/derived/`).


In [ ]:
# Colab T4: bitsandbytes for quantized loads. Restart the runtime if
# bitsandbytes was just installed and the kernel has not picked it up.
import sys
import subprocess
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.44",
        "accelerate>=0.33",
        "bitsandbytes>=0.43",
        "pandas",
        "scipy",
        "networkx",
        "tqdm",
        "huggingface_hub",
    ]
)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ── knobs ────────────────────────────────────────────────────────────────
# Set LIMIT to an int for a smoke test (e.g. 2 items per family). None = full run.
LIMIT = None
DRY_RUN = False          # True: skip GPU, write placeholder rows (pipeline check)
RESUME = True

# Private GitHub clone (Colab secret GITHUB_TOKEN, or env). Public clone works
# without a token. If this notebook is already inside the repo, clone is skipped.
REPO_URL = os.environ.get(
    "RVC_REPO_URL",
    "https://github.com/Adya6714/retrieval-vs-computation.git",
)
REPO_COMMIT = os.environ.get("RVC_REPO_COMMIT", "")  # empty = default branch HEAD

def _secret(name: str) -> str:
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name) or ""
    except Exception:
        return ""

HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN")
GH_TOKEN = _secret("GITHUB_TOKEN")

# Llama-3.1-8B-Instruct is gated: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as _hf_exc:
        print("[setup] huggingface login skipped:", _hf_exc)

def _looks_like_repo(p: Path) -> bool:
    return (p / "probes" / "contamination" / "verify.py").is_file() and (
        p / "data" / "problems" / "question_bank_gsm.csv"
    ).is_file()

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if _looks_like_repo(cand):
            return cand
    colab = Path("/content/retrieval-vs-computation")
    if _looks_like_repo(colab):
        return colab
    return colab

REPO_ROOT = _find_repo()
if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    url = REPO_URL
    if GH_TOKEN and "github.com" in url and url.startswith("https://"):
        url = url.replace("https://", f"https://{GH_TOKEN}@")
    print(f"[setup] cloning {REPO_URL} → {REPO_ROOT}")
    cmd = ["git", "clone", "--depth", "1", url, str(REPO_ROOT)]
    subprocess.check_call(cmd)
    if REPO_COMMIT:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REPO_COMMIT])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_COMMIT])

assert _looks_like_repo(REPO_ROOT), (
    f"Could not find probes/ + question banks under {REPO_ROOT}. "
    "Clone the retrieval-vs-computation repo, or set RVC_REPO_URL / GITHUB_TOKEN."
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT_DIR = Path("/content/colab_out") if Path("/content").exists() else (REPO_ROOT / "colab_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[setup] REPO_ROOT={REPO_ROOT}")
print(f"[setup] OUT_DIR={OUT_DIR}")
print(f"[setup] LIMIT={LIMIT} DRY_RUN={DRY_RUN} RESUME={RESUME}")


# ═══════════════════════════════════════════════════════════════════════════
# K-suite master knobs (edit here; then Runtime → Run all)
# ═══════════════════════════════════════════════════════════════════════════
# Smoke first: LIMIT=2, DRY_RUN=True  →  then LIMIT=None, DRY_RUN=False
# Shared LIMIT/DRY_RUN/RESUME come from SETUP_REPO above.

RUN_K1_O5 = True       # teacher-forced likelihood (~4299 rows; Qwen primary)
RUN_K2_O6 = True       # quantization sensitivity (60 items; after K1)
RUN_K3_O15 = True      # surprisal contamination
RUN_K4_O8 = True       # mech↔behavior link (needs K1 CSV + O7 PASS)
RUN_K6_O14B = True     # naming likelihood (Qwen 1.5B/3B)
RUN_K7_DS16 = True     # recognition vs recall
RUN_K5_O16 = True      # open-model calibration — run last (sparse GT caveat)

# K1: skip gated Llama when no HF_TOKEN (Qwen arms are primary)
SKIP_LLAMA = None      # None = auto (skip if HF_TOKEN missing); True/False force

# K3: first pass = fp16 primaries only
RUN_OPTIONAL_SCALE = False

# K4: ALGO + GSM (O7 PASS already recorded for GSM)
INCLUDE_GSM = True

# K6 / K7 smoke overrides (None = full)
LIMIT_PAIRS = None           # O14b pairs (×3 arms)
LIMIT_PER_FAMILY = None      # DS16 problem_ids per family
K_DISTRACTORS = 4
MAX_NEW_TOKENS_BW = 512      # O14b greedy
N_BOOT = 5000
SEED = 42
FLOOR_ACC = 0.05
MIN_K_PCT = 20
DS16_MAX_NEW_TOKENS = {"GSM": 256, "ALGO": 192, "BW": 512}
VARIANTS = ("canonical", "W3")  # DS16 default; other arms redefine locally

if SKIP_LLAMA is None:
    SKIP_LLAMA = not bool(HF_TOKEN)
print("[suite] RUN flags:",
      {k: v for k, v in globals().items() if k.startswith("RUN_K")})
print(f"[suite] SKIP_LLAMA={SKIP_LLAMA}  RUN_OPTIONAL_SCALE={RUN_OPTIONAL_SCALE}  "
      f"INCLUDE_GSM={INCLUDE_GSM}  LIMIT={LIMIT}  DRY_RUN={DRY_RUN}")


---
# K1 — O5 teacher-forced likelihood

Unblocks C6, O8, and idle cells. ~full P1 grid × models. `SKIP_LLAMA` drops gated Llama when no token.

**Flag:** `RUN_K1_O5` · **Outputs (download separately):** `O5_teacher_forced_likelihood.csv`


<details><summary>Arm README (from standalone notebook)</summary>

# O5 — Teacher-forced gold-sequence likelihood (Probe-1 grid)

**Floor-free robustness measure.** Binary retention is undefined at floor and ceiling; the 0.30 accuracy floor suppresses most retention cells; N3 collapsed when W3 accuracy was 1/60 and 0/61. Log-likelihood of the gold answer is continuous, defined everywhere, and deterministic (no sampling noise).

**Models (HuggingFace, Colab T4):**
| Model | Load | Role |
|-------|------|------|
| `Qwen/Qwen2.5-1.5B-Instruct` | fp16, `attn_implementation="sdpa"` | primary |
| `Qwen/Qwen2.5-3B-Instruct` | fp16, `attn_implementation="sdpa"` | primary |
| `meta-llama/Llama-3.1-8B-Instruct` | 4-bit NF4, `compute_dtype=float16`, sdpa | robustness check only |

**T4 hard constraints:** fp16 only (no bf16), no FlashAttention-2, `attn_implementation="sdpa"`.

**Grid:** every `(family ∈ {GSM, ALGO, BW}, problem_id, variant ∈ {canonical, W1…W6}, model)` present in the question banks.

**Prompt:** identical Appendix-N Probe-1 template used by the other Colab Probe-1 notebooks (`PROBE1_TEMPLATE` + `FAMILY_FORMAT` + chat template). Do not rewrite.

**Output:** `colab_out/O5_teacher_forced_likelihood.csv` — **per-item rows only**. No aggregates here (analysis is O10).

**Secrets:** `HF_TOKEN` (gated Llama); optional `GITHUB_TOKEN` if the repo is private.

---

### CRITICAL CAVEAT — gold TARGET STRING can change under W3 / W6

W3 renames entities and W6 regenerates parameters, so the gold **target string** often differs from canonical. A raw Δ mean_logprob then confounds *"the model's belief moved"* with *"we are scoring a different string."*

This notebook handles that as follows:
1. **Primary metric is always `mean_logprob`** (length-normalized). Persist `sum_logprob` but do not treat it as the comparison unit.
2. **`target_identical`:** True iff normalized variant gold == normalized canonical gold (cleanest Δ cells; report separately in O10). Checked for all variants; especially informative for W1/W2/W4/W5.
3. **`target_comparable` + `control_*`:** when the variant gold is well-formed under the **canonical** prompt, also teacher-force that gold under the canonical prompt and persist `control_mean_logprob` (etc.). When impossible, `target_comparable=False` and control fields are empty.
4. Well-formed rule (documented, deterministic): identical golds → comparable; else GSM numeric golds remain format-valid under any GSM prompt → comparable; ALGO/BW with a **different** gold → not comparable (entity/operator/instance mismatch).

</details>


## Item queue — full Probe-1 bank grid + clone families

Loads `data/problems/question_bank_{gsm,algo,bw}.csv`. Every bank row is one cell. Clone IDs come from `probes.common.clones` / `bank_clone_audit.csv` (ALGO); GSM/BW use `SINGLETON_{problem_id}`.


In [ ]:
from __future__ import annotations
if not (RUN_K1_O5):
    print('[skip] K1_O5  (RUN_K1_O5=False)')
else:

    import csv
    import gc
    import re
    from typing import Any

    import pandas as pd
    import torch
    import torch.nn.functional as F
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    from probes.common.clones import algo_cluster_map

    # Same Appendix-N template as llama_greedy_behavioural / mechanistic notebooks.
    PROBE1_TEMPLATE = (
        "Solve the following problem exactly and provide only the final answer "
        "in the required output format. Problem: {problem}. Format instruction: "
        "{family_specific_output_format}."
    )

    FAMILY_FORMAT = {
        "GSM": (
            "Write the final numerical answer on its own line as #### <number>. "
            "No other text after that tag."
        ),
        "ALGO": (
            "Follow the problem's required output format exactly "
            "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
        ),
        "BW": (
            "A numbered list of actions only. Each action must be one of the "
            "permitted operators with their arguments. No explanation."
        ),
    }

    VARIANTS = ("canonical", "W1", "W2", "W3", "W4", "W5", "W6")

    MODELS: list[tuple[str, str]] = [
        ("Qwen/Qwen2.5-1.5B-Instruct", "fp16"),
        ("Qwen/Qwen2.5-3B-Instruct", "fp16"),
        ("meta-llama/Llama-3.1-8B-Instruct", "nf4"),  # robustness check only
    ]

    O5_CSV = OUT_DIR / "O5_teacher_forced_likelihood.csv"

    OUT_COLUMNS = [
        "family",
        "problem_id",
        "variant",
        "model",
        "n_gold_tokens",
        "sum_logprob",
        "mean_logprob",
        "gold_first_token_rank",
        "gold_first_token_logprob",
        "prompt_n_tokens",
        "clone_family",
        "target_identical",
        "target_comparable",
        "control_n_gold_tokens",
        "control_sum_logprob",
        "control_mean_logprob",
        "control_gold_first_token_rank",
        "control_gold_first_token_logprob",
    ]


    def _norm_vt(v: str) -> str:
        v = str(v).strip()
        return "canonical" if v.lower() == "canonical" else v.upper()


    def _strip_csv_quotes(text: str) -> str:
        s = str(text)
        if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
            s = s[1:-1]
        return s


    def norm_gold(text: str) -> str:
        """Whitespace-normalized gold for identity checks (not for tokenization)."""
        lines = [ln.strip() for ln in str(text).replace("\r\n", "\n").split("\n")]
        return "\n".join(ln for ln in lines if ln)


    def looks_numeric_gold(text: str) -> bool:
        s = norm_gold(text)
        s = re.sub(r"^####\s*", "", s).replace(",", "").strip()
        if not s:
            return False
        try:
            float(s)
            return True
        except ValueError:
            return bool(re.fullmatch(r"-?\d+(?:\.\d+)?", s))


    def build_prompt(problem_text: str, family: str) -> str:
        """Identical Probe-1 user string construction as the behavioural Colab notebooks."""
        return PROBE1_TEMPLATE.format(
            problem=problem_text.strip(),
            family_specific_output_format=FAMILY_FORMAT[family],
        )


    def _load_bank(path: Path) -> pd.DataFrame:
        df = pd.read_csv(path, dtype=str).fillna("")
        df["problem_id"] = df["problem_id"].astype(str).str.strip()
        df["variant_type"] = df["variant_type"].map(_norm_vt)
        df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
        df["correct_answer"] = df["correct_answer"].map(_strip_csv_quotes)
        return df


    def clone_family_for(family: str, problem_id: str, cmap: dict[str, str]) -> str:
        if family == "ALGO":
            return cmap.get(problem_id, f"SINGLETON_{problem_id}")
        return f"SINGLETON_{problem_id}"


    def target_flags(
        family: str,
        variant: str,
        can_gold: str,
        var_gold: str,
    ) -> tuple[bool, bool]:
        """Return (target_identical, target_comparable).

        Comparable ⇒ we may teacher-force *variant gold* under the *canonical* prompt.
        """
        identical = norm_gold(can_gold) == norm_gold(var_gold)
        if identical:
            return True, True
        # Different string: only GSM keeps a format-valid numeric answer under can-prompt.
        if family == "GSM" and looks_numeric_gold(var_gold):
            return False, True
        # W3/W6 (and non-identical ALGO/BW W*) change entities/operators/instance.
        _ = variant  # retained for callers / future tightening
        return False, False


    def load_items(limit: int | None) -> list[dict[str, Any]]:
        specs = [
            ("GSM", REPO_ROOT / "data/problems/question_bank_gsm.csv"),
            ("ALGO", REPO_ROOT / "data/problems/question_bank_algo.csv"),
            ("BW", REPO_ROOT / "data/problems/question_bank_bw.csv"),
        ]
        cmap = algo_cluster_map()
        items: list[dict[str, Any]] = []
        for family, path in specs:
            df = _load_bank(path)
            can = {
                str(r.problem_id): str(r.correct_answer)
                for r in df.loc[df.variant_type == "canonical"].itertuples(index=False)
            }
            can_text = {
                str(r.problem_id): str(r.problem_text)
                for r in df.loc[df.variant_type == "canonical"].itertuples(index=False)
            }
            for _, row in df.iterrows():
                pid = str(row["problem_id"])
                vt = str(row["variant_type"])
                if vt not in VARIANTS:
                    continue
                if pid not in can:
                    continue
                var_gold = str(row["correct_answer"])
                c_gold = can[pid]
                identical, comparable = target_flags(family, vt, c_gold, var_gold)
                items.append(
                    {
                        "family": family,
                        "problem_id": pid,
                        "variant": vt,
                        "problem_text": str(row["problem_text"]),
                        "gold": var_gold,
                        "canonical_problem_text": can_text[pid],
                        "canonical_gold": c_gold,
                        "target_identical": identical,
                        "target_comparable": comparable and vt != "canonical",
                        "clone_family": clone_family_for(family, pid, cmap),
                    }
                )
        if limit is not None:
            # Keep a balanced smoke slice: first `limit` IDs per family × all their variants.
            keep: set[tuple[str, str]] = set()
            for fam in ("GSM", "ALGO", "BW"):
                ids = sorted({x["problem_id"] for x in items if x["family"] == fam})[:limit]
                keep |= {(fam, pid) for pid in ids}
            items = [x for x in items if (x["family"], x["problem_id"]) in keep]
        return items


    ITEMS = load_items(LIMIT)
    print(f"[queue] {len(ITEMS)} cells (LIMIT={LIMIT})")
    print(pd.DataFrame(ITEMS).groupby(["family", "variant"]).size().unstack(fill_value=0).to_string())
    print(
        "[caveat] target_identical rates by variant:\n",
        pd.DataFrame(ITEMS)
        .groupby("variant")["target_identical"]
        .mean()
        .reindex(list(VARIANTS))
        .round(3)
        .to_string(),
    )
    print(
        "[caveat] target_comparable rates by variant:\n",
        pd.DataFrame(ITEMS)
        .groupby("variant")["target_comparable"]
        .mean()
        .reindex(list(VARIANTS))
        .round(3)
        .to_string(),
    )

    # Suite gate: drop Llama when SKIP_LLAMA (needs HF_TOKEN for gated weights)
    if SKIP_LLAMA:
        _before = list(MODELS)
        MODELS = [
            m for m in MODELS
            if "llama" not in (m[0] if isinstance(m, (tuple, list)) else str(m)).lower()
        ]
        print(f"[suite] SKIP_LLAMA filtered MODELS {_before} → {MODELS}")


## Teacher-forced likelihood

For each cell: chat-wrap the Probe-1 user prompt, append the bank gold string (verbatim `correct_answer`), run one forward pass, and sum token log-probs of the gold continuation.

Primary fields: `sum_logprob`, `mean_logprob = sum / n_gold_tokens`, `gold_first_token_rank`, `gold_first_token_logprob`.

When `target_comparable`, also score **variant gold under the canonical prompt** → `control_*` columns.


In [ ]:
if not (RUN_K1_O5):
    print('[skip] K1_O5  (RUN_K1_O5=False)')
else:
    def wrap_chat(tokenizer, user_text: str) -> str:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_text}],
            add_generation_prompt=True,
            tokenize=False,
        )


    def resolve_continuation(
        tokenizer,
        prompt: str,
        answer: str,
    ) -> tuple[list[int], list[int], str]:
        """Prompt-aware gold token ids (joint encode; try '' then ' ' separator)."""

        def enc(text: str) -> list[int]:
            return tokenizer.encode(text, add_special_tokens=False)

        prompt_ids = enc(prompt)
        answer = str(answer)
        if not answer:
            return prompt_ids, [], "EMPTY"
        candidates: list[tuple[str, list[int], int]] = []
        for sep in ("", " "):
            joint = enc(prompt + sep + answer)
            if len(joint) <= len(prompt_ids):
                continue
            if joint[: len(prompt_ids)] != prompt_ids:
                continue
            rest = joint[len(prompt_ids) :]
            candidates.append((sep, rest, len(joint)))
        if not candidates:
            bare = enc(answer)
            return prompt_ids, bare, "FALLBACK"
        candidates.sort(key=lambda c: c[2])
        sep, rest, _ = candidates[0]
        return prompt_ids, rest, repr(sep)


    @torch.inference_mode()
    def teacher_forced_metrics(
        model,
        tokenizer,
        device,
        user_text: str,
        gold_text: str,
    ) -> dict[str, Any]:
        prompt = wrap_chat(tokenizer, user_text)
        prompt_ids, gold_ids, sep_note = resolve_continuation(tokenizer, prompt, gold_text)
        n_prompt = len(prompt_ids)
        n_gold = len(gold_ids)
        if n_gold == 0:
            return {
                "n_gold_tokens": 0,
                "sum_logprob": float("nan"),
                "mean_logprob": float("nan"),
                "gold_first_token_rank": -1,
                "gold_first_token_logprob": float("nan"),
                "prompt_n_tokens": n_prompt,
                "sep_note": sep_note,
            }
        if DRY_RUN or model is None:
            return {
                "n_gold_tokens": n_gold,
                "sum_logprob": 0.0,
                "mean_logprob": 0.0,
                "gold_first_token_rank": 1,
                "gold_first_token_logprob": 0.0,
                "prompt_n_tokens": n_prompt,
                "sep_note": "DRY_RUN",
            }

        input_ids = torch.tensor([prompt_ids + gold_ids], dtype=torch.long, device=device)
        out = model(input_ids=input_ids, use_cache=False)
        # logits[t] predicts token t+1
        logits = out.logits[0]  # [seq, vocab]
        # gold token at absolute index n_prompt + i is predicted by position n_prompt + i - 1
        gold_logits = logits[n_prompt - 1 : n_prompt + n_gold - 1]
        log_probs = F.log_softmax(gold_logits.float(), dim=-1)
        gold_t = torch.tensor(gold_ids, device=device, dtype=torch.long)
        tok_lp = log_probs.gather(1, gold_t.unsqueeze(1)).squeeze(1)
        sum_lp = float(tok_lp.sum().item())
        mean_lp = sum_lp / n_gold

        first_logits = gold_logits[0].float()
        first_tid = int(gold_ids[0])
        first_lp = float(F.log_softmax(first_logits, dim=-1)[first_tid].item())
        rank = int((first_logits > first_logits[first_tid]).sum().item()) + 1

        del out, logits, input_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "n_gold_tokens": n_gold,
            "sum_logprob": round(sum_lp, 6),
            "mean_logprob": round(mean_lp, 6),
            "gold_first_token_rank": rank,
            "gold_first_token_logprob": round(first_lp, 6),
            "prompt_n_tokens": n_prompt,
            "sep_note": sep_note,
        }


    def load_model(model_id: str, quant: str):
        assert torch.cuda.is_available() or DRY_RUN, "GPU required (Colab T4) unless DRY_RUN."
        tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True)
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        if DRY_RUN:
            print(f"[model] DRY_RUN skip load: {model_id} ({quant})")
            return tok, None, torch.device("cpu")

        common = dict(
            device_map="auto",
            token=HF_TOKEN or True,
            attn_implementation="sdpa",  # T4: no FlashAttention-2
            torch_dtype=torch.float16,  # T4: fp16 only, no bf16
        )
        if quant == "fp16":
            mdl = AutoModelForCausalLM.from_pretrained(model_id, **common)
            label = "fp16 unquantized + sdpa"
        elif quant == "nf4":
            bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb,
                **common,
            )
            label = "nf4 4-bit bitsandbytes (compute_dtype=float16) + sdpa"
        else:
            raise ValueError(quant)
        mdl.eval()
        device = next(mdl.parameters()).device
        print(f"[model] {model_id}  {label}  device={device}")
        return tok, mdl, device


    def unload(mdl):
        if mdl is None:
            return
        del mdl
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    def empty_control() -> dict[str, Any]:
        return {
            "control_n_gold_tokens": "",
            "control_sum_logprob": "",
            "control_mean_logprob": "",
            "control_gold_first_token_rank": "",
            "control_gold_first_token_logprob": "",
        }


    def score_item(model, tokenizer, device, item: dict, model_id: str) -> dict[str, Any]:
        user = build_prompt(item["problem_text"], item["family"])
        primary = teacher_forced_metrics(model, tokenizer, device, user, item["gold"])
        row: dict[str, Any] = {
            "family": item["family"],
            "problem_id": item["problem_id"],
            "variant": item["variant"],
            "model": model_id,
            "n_gold_tokens": primary["n_gold_tokens"],
            "sum_logprob": primary["sum_logprob"],
            "mean_logprob": primary["mean_logprob"],
            "gold_first_token_rank": primary["gold_first_token_rank"],
            "gold_first_token_logprob": primary["gold_first_token_logprob"],
            "prompt_n_tokens": primary["prompt_n_tokens"],
            "clone_family": item["clone_family"],
            "target_identical": bool(item["target_identical"]),
            "target_comparable": bool(item["target_comparable"]),
        }
        row.update(empty_control())
        if item["target_comparable"]:
            can_user = build_prompt(item["canonical_problem_text"], item["family"])
            # Control: W-variant gold under the canonical prompt (same gold string, different surface).
            ctrl = teacher_forced_metrics(model, tokenizer, device, can_user, item["gold"])
            row["control_n_gold_tokens"] = ctrl["n_gold_tokens"]
            row["control_sum_logprob"] = ctrl["sum_logprob"]
            row["control_mean_logprob"] = ctrl["mean_logprob"]
            row["control_gold_first_token_rank"] = ctrl["gold_first_token_rank"]
            row["control_gold_first_token_logprob"] = ctrl["gold_first_token_logprob"]
        return row


## Run — write every per-item row (resume-safe)

No aggregate statistics. Resume key: `(model, family, problem_id, variant)`.


In [ ]:
if not (RUN_K1_O5):
    print('[skip] K1_O5  (RUN_K1_O5=False)')
else:
    def _done_keys(path: Path) -> set[tuple[str, str, str, str]]:
        if not path.exists():
            return set()
        df = pd.read_csv(path, dtype=str)
        need = {"model", "family", "problem_id", "variant"}
        if not need.issubset(df.columns):
            return set()
        return {
            (str(r.model), str(r.family), str(r.problem_id), str(r.variant))
            for r in df.itertuples(index=False)
        }


    def append_rows(path: Path, rows: list[dict[str, Any]]) -> None:
        if not rows:
            return
        write_header = not path.exists()
        with path.open("a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=OUT_COLUMNS, extrasaction="ignore")
            if write_header:
                w.writeheader()
            for r in rows:
                w.writerow({k: r.get(k, "") for k in OUT_COLUMNS})


    done = _done_keys(O5_CSV) if RESUME else set()
    print(f"[resume] {len(done)} rows already in {O5_CSV}")

    for model_id, quant in MODELS:
        pending = [
            it
            for it in ITEMS
            if (model_id, it["family"], it["problem_id"], it["variant"]) not in done
        ]
        print(f"\n=== {model_id} ({quant})  pending={len(pending)}/{len(ITEMS)} ===")
        if not pending:
            continue
        tok, mdl, device = load_model(model_id, quant)
        buf: list[dict[str, Any]] = []
        try:
            for it in tqdm(pending, desc=model_id.split("/")[-1]):
                row = score_item(mdl, tok, device, it, model_id)
                buf.append(row)
                if len(buf) >= 25:
                    append_rows(O5_CSV, buf)
                    buf.clear()
            append_rows(O5_CSV, buf)
        finally:
            unload(mdl)

    print(f"\n[done] wrote {O5_CSV}")
    if O5_CSV.exists():
        out_df = pd.read_csv(O5_CSV)
        print(f"[done] n_rows={len(out_df)}  models={sorted(out_df['model'].unique())}")
        print(out_df.groupby(["model", "family"]).size().unstack(fill_value=0).to_string())
        # Intentionally no mean/retention aggregates — O10 owns analysis.


### Download — K1_O5 only


In [ ]:
if not (RUN_K1_O5):
    print("[skip download] K1_O5")
else:
    from pathlib import Path as _P
    import shutil as _shutil
    _names = ['O5_teacher_forced_likelihood.csv']
    _paths = [OUT_DIR / n for n in _names]
    _present = [p for p in _paths if p.is_file()]
    print(f"[download K1_O5] present={len(_present)}/{len(_paths)} in {OUT_DIR}")
    for p in _paths:
        print(" ", "OK" if p.is_file() else "MISSING", p.name)
    _drive_dir = _P("/content/drive/MyDrive/rvc_colab_out")
    if _P("/content").exists() and not _P("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
        except Exception as _exc:
            print("[drive] mount skipped:", _exc)
    try:
        _drive_dir.mkdir(parents=True, exist_ok=True)
        for p in _present:
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
    except Exception as _exc:
        print("[backup] skipped:", _exc)
    try:
        from google.colab import files as _colab_files  # type: ignore
        for p in _present:
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
    except Exception as _exc:
        print("[download] skipped (not Colab or blocked):", _exc)


---
# K2 — O6 quantization sensitivity

60 stratified items on Qwen2.5-1.5B. Run right after K1.

**Flag:** `RUN_K2_O6` · **Outputs (download separately):** `O6_quantization_sensitivity.csv`, `O6_quantization_sensitivity_items.csv`, `O6_quantization_sensitivity_summary.txt`, `O6_subsample_manifest.json`


<details><summary>Arm README (from standalone notebook)</summary>

# O6 — Quantization sensitivity bound (O5 measurement)

Quantization perturbs logits. O5's `gold_first_token_rank` is a rank over the full vocabulary, so **4-bit ranks may not be comparable to fp16**. This notebook bounds the error on Qwen2.5-1.5B-Instruct.

**Design**
1. Draw a **fixed-seed stratified subsample of 60** Probe-1 cells (20 GSM + 20 ALGO + 20 BW) from the same O5 item universe.
2. Run the **identical O5 teacher-forced gold-sequence measurement** at three precisions on `Qwen/Qwen2.5-1.5B-Instruct`:
   - **fp16** (reference)
   - **8-bit** (bitsandbytes)
   - **4-bit NF4** (`compute_dtype=float16`)
3. For each precision pair, report: mean |Δ mean_logprob|, Spearman(mean_logprob), median |Δ gold_first_token_rank|, 95th percentile |Δ rank|.

**T4:** fp16 only for unquantized loads; `attn_implementation="sdpa"`; no FlashAttention-2; no bf16.

**Decision rule (methods):** if the **median absolute rank shift** for fp16↔4-bit exceeds roughly **50** positions, drop all 4-bit rank measurements from the paper and keep only fp16 (state this explicitly).

**Outputs**
- `colab_out/O6_quantization_sensitivity.csv` — pairwise summary (required)
- `colab_out/O6_quantization_sensitivity_items.csv` — every per-item score (audit; do not lose intermediates)
- `colab_out/O6_quantization_sensitivity_summary.txt` — one-paragraph usability verdict

`LIMIT` (setup cell) shrinks the per-family draw for smoke tests (e.g. `LIMIT=2` → 6 items).

</details>


## Stratified 60-item subsample + O5 measurement primitives

Same Appendix-N prompt and teacher-forced gold continuation as O5. Subsample seed is fixed (`SUBSAMPLE_SEED=42`).


In [ ]:
from __future__ import annotations
if not (RUN_K2_O6):
    print('[skip] K2_O6  (RUN_K2_O6=False)')
else:

    import csv
    import gc
    import json
    import random
    import re
    from typing import Any

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from scipy import stats
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    PRECISIONS = ("fp16", "int8", "nf4")
    SUBSAMPLE_SEED = 42
    N_PER_FAMILY = 20  # 20 × 3 = 60; overridden downward when LIMIT is set
    RANK_DROP_THRESHOLD = 50  # median |Δ rank| fp16↔nf4; methods kill criterion

    PROBE1_TEMPLATE = (
        "Solve the following problem exactly and provide only the final answer "
        "in the required output format. Problem: {problem}. Format instruction: "
        "{family_specific_output_format}."
    )
    FAMILY_FORMAT = {
        "GSM": (
            "Write the final numerical answer on its own line as #### <number>. "
            "No other text after that tag."
        ),
        "ALGO": (
            "Follow the problem's required output format exactly "
            "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
        ),
        "BW": (
            "A numbered list of actions only. Each action must be one of the "
            "permitted operators with their arguments. No explanation."
        ),
    }
    VARIANTS = ("canonical", "W1", "W2", "W3", "W4", "W5", "W6")

    O6_ITEMS_CSV = OUT_DIR / "O6_quantization_sensitivity_items.csv"
    O6_CSV = OUT_DIR / "O6_quantization_sensitivity.csv"
    O6_SUMMARY_TXT = OUT_DIR / "O6_quantization_sensitivity_summary.txt"

    ITEM_COLUMNS = [
        "family", "problem_id", "variant", "precision", "model",
        "n_gold_tokens", "sum_logprob", "mean_logprob",
        "gold_first_token_rank", "gold_first_token_logprob", "prompt_n_tokens",
    ]
    PAIR_COLUMNS = [
        "precision_a", "precision_b", "n_items",
        "mean_abs_diff_mean_logprob", "spearman_mean_logprob", "spearman_pvalue",
        "median_abs_rank_shift", "p95_abs_rank_shift",
        "max_abs_rank_shift",
    ]


    def _norm_vt(v: str) -> str:
        v = str(v).strip()
        return "canonical" if v.lower() == "canonical" else v.upper()


    def _strip_csv_quotes(text: str) -> str:
        s = str(text)
        if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
            s = s[1:-1]
        return s


    def build_prompt(problem_text: str, family: str) -> str:
        return PROBE1_TEMPLATE.format(
            problem=problem_text.strip(),
            family_specific_output_format=FAMILY_FORMAT[family],
        )


    def _load_bank(path: Path) -> pd.DataFrame:
        df = pd.read_csv(path, dtype=str).fillna("")
        df["problem_id"] = df["problem_id"].astype(str).str.strip()
        df["variant_type"] = df["variant_type"].map(_norm_vt)
        df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
        df["correct_answer"] = df["correct_answer"].map(_strip_csv_quotes)
        return df


    def load_o5_universe() -> list[dict[str, Any]]:
        """Full O5 cell universe (family × problem × variant present in banks)."""
        specs = [
            ("GSM", REPO_ROOT / "data/problems/question_bank_gsm.csv"),
            ("ALGO", REPO_ROOT / "data/problems/question_bank_algo.csv"),
            ("BW", REPO_ROOT / "data/problems/question_bank_bw.csv"),
        ]
        items: list[dict[str, Any]] = []
        for family, path in specs:
            df = _load_bank(path)
            can_ids = set(df.loc[df.variant_type == "canonical", "problem_id"])
            for _, row in df.iterrows():
                pid = str(row["problem_id"])
                vt = str(row["variant_type"])
                if vt not in VARIANTS or pid not in can_ids:
                    continue
                items.append(
                    {
                        "family": family,
                        "problem_id": pid,
                        "variant": vt,
                        "problem_text": str(row["problem_text"]),
                        "gold": str(row["correct_answer"]),
                    }
                )
        return items


    def stratified_subsample(
        universe: list[dict[str, Any]],
        n_per_family: int,
        seed: int,
    ) -> list[dict[str, Any]]:
        rng = random.Random(seed)
        out: list[dict[str, Any]] = []
        for fam in ("GSM", "ALGO", "BW"):
            pool = [x for x in universe if x["family"] == fam]
            if not pool:
                raise RuntimeError(f"empty pool for {fam}")
            k = min(n_per_family, len(pool))
            out.extend(rng.sample(pool, k=k))
        out.sort(key=lambda x: (x["family"], x["problem_id"], x["variant"]))
        return out


    _n_per = N_PER_FAMILY if LIMIT is None else min(int(LIMIT), N_PER_FAMILY)
    UNIVERSE = load_o5_universe()
    ITEMS = stratified_subsample(UNIVERSE, _n_per, SUBSAMPLE_SEED)
    print(
        f"[sample] n={len(ITEMS)}  seed={SUBSAMPLE_SEED}  "
        f"n_per_family={_n_per}  universe={len(UNIVERSE)}"
    )
    print(pd.DataFrame(ITEMS).groupby(["family", "variant"]).size().unstack(fill_value=0).to_string())
    (OUT_DIR / "O6_subsample_manifest.json").write_text(
        json.dumps(
            {
                "model": MODEL_ID,
                "seed": SUBSAMPLE_SEED,
                "n_per_family": _n_per,
                "n_items": len(ITEMS),
                "items": [
                    {"family": x["family"], "problem_id": x["problem_id"], "variant": x["variant"]}
                    for x in ITEMS
                ],
            },
            indent=2,
        )
    )
    print(f"[sample] wrote {OUT_DIR / 'O6_subsample_manifest.json'}")


## Teacher-forced metrics (identical to O5) + precision loaders


In [ ]:
if not (RUN_K2_O6):
    print('[skip] K2_O6  (RUN_K2_O6=False)')
else:
    def wrap_chat(tokenizer, user_text: str) -> str:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_text}],
            add_generation_prompt=True,
            tokenize=False,
        )


    def resolve_continuation(
        tokenizer,
        prompt: str,
        answer: str,
    ) -> tuple[list[int], list[int], str]:
        def enc(text: str) -> list[int]:
            return tokenizer.encode(text, add_special_tokens=False)

        prompt_ids = enc(prompt)
        answer = str(answer)
        if not answer:
            return prompt_ids, [], "EMPTY"
        candidates: list[tuple[str, list[int], int]] = []
        for sep in ("", " "):
            joint = enc(prompt + sep + answer)
            if len(joint) <= len(prompt_ids):
                continue
            if joint[: len(prompt_ids)] != prompt_ids:
                continue
            rest = joint[len(prompt_ids) :]
            candidates.append((sep, rest, len(joint)))
        if not candidates:
            return prompt_ids, enc(answer), "FALLBACK"
        candidates.sort(key=lambda c: c[2])
        sep, rest, _ = candidates[0]
        return prompt_ids, rest, repr(sep)


    @torch.inference_mode()
    def teacher_forced_metrics(
        model,
        tokenizer,
        device,
        user_text: str,
        gold_text: str,
    ) -> dict[str, Any]:
        prompt = wrap_chat(tokenizer, user_text)
        prompt_ids, gold_ids, sep_note = resolve_continuation(tokenizer, prompt, gold_text)
        n_prompt = len(prompt_ids)
        n_gold = len(gold_ids)
        if n_gold == 0:
            return {
                "n_gold_tokens": 0,
                "sum_logprob": float("nan"),
                "mean_logprob": float("nan"),
                "gold_first_token_rank": -1,
                "gold_first_token_logprob": float("nan"),
                "prompt_n_tokens": n_prompt,
                "sep_note": sep_note,
            }
        if DRY_RUN or model is None:
            # Deterministic fake ranks keyed by precision label length — only for pipeline checks.
            return {
                "n_gold_tokens": n_gold,
                "sum_logprob": -0.1 * n_gold,
                "mean_logprob": -0.1,
                "gold_first_token_rank": 1,
                "gold_first_token_logprob": -0.1,
                "prompt_n_tokens": n_prompt,
                "sep_note": "DRY_RUN",
            }

        input_ids = torch.tensor([prompt_ids + gold_ids], dtype=torch.long, device=device)
        out = model(input_ids=input_ids, use_cache=False)
        logits = out.logits[0]
        gold_logits = logits[n_prompt - 1 : n_prompt + n_gold - 1]
        log_probs = F.log_softmax(gold_logits.float(), dim=-1)
        gold_t = torch.tensor(gold_ids, device=device, dtype=torch.long)
        tok_lp = log_probs.gather(1, gold_t.unsqueeze(1)).squeeze(1)
        sum_lp = float(tok_lp.sum().item())
        mean_lp = sum_lp / n_gold
        first_logits = gold_logits[0].float()
        first_tid = int(gold_ids[0])
        first_lp = float(F.log_softmax(first_logits, dim=-1)[first_tid].item())
        rank = int((first_logits > first_logits[first_tid]).sum().item()) + 1
        del out, logits, input_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "n_gold_tokens": n_gold,
            "sum_logprob": round(sum_lp, 6),
            "mean_logprob": round(mean_lp, 6),
            "gold_first_token_rank": rank,
            "gold_first_token_logprob": round(first_lp, 6),
            "prompt_n_tokens": n_prompt,
            "sep_note": sep_note,
        }


    def load_model(precision: str):
        assert torch.cuda.is_available() or DRY_RUN, "GPU required (Colab T4) unless DRY_RUN."
        tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN or True)
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        if DRY_RUN:
            print(f"[model] DRY_RUN skip load: {MODEL_ID} ({precision})")
            return tok, None, torch.device("cpu")

        common = dict(
            device_map="auto",
            token=HF_TOKEN or True,
            attn_implementation="sdpa",
        )
        if precision == "fp16":
            mdl = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                torch_dtype=torch.float16,
                **common,
            )
            label = "fp16 unquantized + sdpa"
        elif precision == "int8":
            bnb = BitsAndBytesConfig(load_in_8bit=True)
            mdl = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                torch_dtype=torch.float16,
                **common,
            )
            label = "int8 bitsandbytes + sdpa"
        elif precision == "nf4":
            bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            mdl = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                torch_dtype=torch.float16,
                **common,
            )
            label = "nf4 4-bit (compute_dtype=float16) + sdpa"
        else:
            raise ValueError(precision)
        mdl.eval()
        device = next(mdl.parameters()).device
        print(f"[model] {MODEL_ID}  {label}  device={device}")
        return tok, mdl, device


    def unload(mdl):
        if mdl is None:
            return
        del mdl
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    def score_primary(model, tokenizer, device, item: dict, precision: str) -> dict[str, Any]:
        user = build_prompt(item["problem_text"], item["family"])
        m = teacher_forced_metrics(model, tokenizer, device, user, item["gold"])
        # DRY_RUN: inject precision-dependent rank noise so pairwise shifts are nonzero in pipeline checks
        if DRY_RUN or model is None:
            bump = {"fp16": 0, "int8": 3, "nf4": 80}[precision]
            m = dict(m)
            m["gold_first_token_rank"] = 1 + bump
            m["mean_logprob"] = round(-0.1 - 0.01 * bump, 6)
            m["sum_logprob"] = round(m["mean_logprob"] * m["n_gold_tokens"], 6)
        return {
            "family": item["family"],
            "problem_id": item["problem_id"],
            "variant": item["variant"],
            "precision": precision,
            "model": MODEL_ID,
            "n_gold_tokens": m["n_gold_tokens"],
            "sum_logprob": m["sum_logprob"],
            "mean_logprob": m["mean_logprob"],
            "gold_first_token_rank": m["gold_first_token_rank"],
            "gold_first_token_logprob": m["gold_first_token_logprob"],
            "prompt_n_tokens": m["prompt_n_tokens"],
        }


## Run all three precisions → pairwise sensitivity + verdict

Decision: if median |Δ rank| for **fp16 vs nf4** > ~50, 4-bit ranks are **not usable** in the paper.


In [ ]:
if not (RUN_K2_O6):
    print('[skip] K2_O6  (RUN_K2_O6=False)')
else:
    def _done_keys(path: Path) -> set[tuple[str, str, str, str]]:
        if not path.exists():
            return set()
        df = pd.read_csv(path, dtype=str)
        need = {"precision", "family", "problem_id", "variant"}
        if not need.issubset(df.columns):
            return set()
        return {
            (str(r.precision), str(r.family), str(r.problem_id), str(r.variant))
            for r in df.itertuples(index=False)
        }


    def append_item_rows(path: Path, rows: list[dict[str, Any]]) -> None:
        if not rows:
            return
        write_header = not path.exists()
        with path.open("a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=ITEM_COLUMNS, extrasaction="ignore")
            if write_header:
                w.writeheader()
            for r in rows:
                w.writerow({k: r.get(k, "") for k in ITEM_COLUMNS})


    done = _done_keys(O6_ITEMS_CSV) if RESUME else set()
    print(f"[resume] {len(done)} item-rows in {O6_ITEMS_CSV}")

    for precision in PRECISIONS:
        pending = [
            it
            for it in ITEMS
            if (precision, it["family"], it["problem_id"], it["variant"]) not in done
        ]
        print(f"\n=== {MODEL_ID} @ {precision}  pending={len(pending)}/{len(ITEMS)} ===")
        if not pending:
            continue
        tok, mdl, device = load_model(precision)
        buf: list[dict[str, Any]] = []
        try:
            for it in tqdm(pending, desc=f"{precision}"):
                buf.append(score_primary(mdl, tok, device, it, precision))
                if len(buf) >= 20:
                    append_item_rows(O6_ITEMS_CSV, buf)
                    buf.clear()
            append_item_rows(O6_ITEMS_CSV, buf)
        finally:
            unload(mdl)

    items_df = pd.read_csv(O6_ITEMS_CSV)
    print(f"[items] n={len(items_df)}  precisions={sorted(items_df.precision.unique())}")


    def pairwise_row(df: pd.DataFrame, a: str, b: str) -> dict[str, Any]:
        wide = (
            df[df.precision.isin([a, b])]
            .pivot_table(
                index=["family", "problem_id", "variant"],
                columns="precision",
                values=["mean_logprob", "gold_first_token_rank"],
                aggfunc="first",
            )
            .dropna()
        )
        # Flatten MultiIndex columns
        lp_a = wide[("mean_logprob", a)].astype(float)
        lp_b = wide[("mean_logprob", b)].astype(float)
        rk_a = wide[("gold_first_token_rank", a)].astype(float)
        rk_b = wide[("gold_first_token_rank", b)].astype(float)
        abs_lp = (lp_a - lp_b).abs()
        abs_rk = (rk_a - rk_b).abs()
        if len(lp_a) >= 2 and lp_a.nunique() > 1 and lp_b.nunique() > 1:
            spearman_r, spearman_p = stats.spearmanr(lp_a, lp_b)
        elif len(lp_a) >= 2:
            spearman_r, spearman_p = float("nan"), float("nan")
        else:
            spearman_r, spearman_p = float("nan"), float("nan")
        return {
            "precision_a": a,
            "precision_b": b,
            "n_items": int(len(wide)),
            "mean_abs_diff_mean_logprob": round(float(abs_lp.mean()), 6),
            "spearman_mean_logprob": (
                round(float(spearman_r), 6) if spearman_r == spearman_r else ""
            ),
            "spearman_pvalue": (
                round(float(spearman_p), 6) if spearman_p == spearman_p else ""
            ),
            "median_abs_rank_shift": round(float(abs_rk.median()), 3),
            "p95_abs_rank_shift": round(float(np.percentile(abs_rk, 95)), 3),
            "max_abs_rank_shift": round(float(abs_rk.max()), 3),
        }


    pairs = [("fp16", "int8"), ("fp16", "nf4"), ("int8", "nf4")]
    pair_rows = [pairwise_row(items_df, a, b) for a, b in pairs]
    pair_df = pd.DataFrame(pair_rows, columns=PAIR_COLUMNS)
    pair_df.to_csv(O6_CSV, index=False)
    print("\n=== O6_quantization_sensitivity.csv ===")
    print(pair_df.to_string(index=False))

    fp16_nf4 = next(r for r in pair_rows if r["precision_a"] == "fp16" and r["precision_b"] == "nf4")
    med = float(fp16_nf4["median_abs_rank_shift"])
    p95 = float(fp16_nf4["p95_abs_rank_shift"])
    mad_lp = float(fp16_nf4["mean_abs_diff_mean_logprob"])
    sp = fp16_nf4["spearman_mean_logprob"]
    n = int(fp16_nf4["n_items"])
    drop_4bit_ranks = med > RANK_DROP_THRESHOLD

    if drop_4bit_ranks:
        verdict = (
            f"On a stratified subsample of {n} Probe-1 cells (seed={SUBSAMPLE_SEED}, "
            f"Qwen2.5-1.5B-Instruct), fp16 vs 4-bit NF4 showed mean |Δ mean_logprob|={mad_lp}, "
            f"Spearman(mean_logprob)={sp}, median |Δ gold_first_token_rank|={med}, "
            f"and 95th-percentile rank shift={p95}. Because the median absolute rank shift "
            f"exceeds ~{RANK_DROP_THRESHOLD} vocabulary positions, 4-bit rank measurements are "
            f"NOT usable for paper claims: drop all 4-bit gold_first_token_rank results and "
            f"retain fp16 (and, where needed, 8-bit) ranks only; state this explicitly in Methods. "
            f"Pairwise table: O6_quantization_sensitivity.csv."
        )
    else:
        verdict = (
            f"On a stratified subsample of {n} Probe-1 cells (seed={SUBSAMPLE_SEED}, "
            f"Qwen2.5-1.5B-Instruct), fp16 vs 4-bit NF4 showed mean |Δ mean_logprob|={mad_lp}, "
            f"Spearman(mean_logprob)={sp}, median |Δ gold_first_token_rank|={med}, "
            f"and 95th-percentile rank shift={p95}. The median absolute rank shift is ≤ "
            f"~{RANK_DROP_THRESHOLD} positions, so 4-bit rank measurements are usable as a "
            f"robustness check alongside fp16 primary results, with the tabulated sensitivity "
            f"bounds reported in Methods. Pairwise table: O6_quantization_sensitivity.csv."
        )

    O6_SUMMARY_TXT.write_text(verdict + "\n")
    print("\n=== SUMMARY (also written to O6_quantization_sensitivity_summary.txt) ===")
    print(verdict)
    print(f"\n[decision] drop_4bit_ranks={drop_4bit_ranks}  threshold={RANK_DROP_THRESHOLD}")


### Download — K2_O6 only


In [ ]:
if not (RUN_K2_O6):
    print("[skip download] K2_O6")
else:
    from pathlib import Path as _P
    import shutil as _shutil
    _names = ['O6_quantization_sensitivity.csv', 'O6_quantization_sensitivity_items.csv', 'O6_quantization_sensitivity_summary.txt', 'O6_subsample_manifest.json']
    _paths = [OUT_DIR / n for n in _names]
    _present = [p for p in _paths if p.is_file()]
    print(f"[download K2_O6] present={len(_present)}/{len(_paths)} in {OUT_DIR}")
    for p in _paths:
        print(" ", "OK" if p.is_file() else "MISSING", p.name)
    _drive_dir = _P("/content/drive/MyDrive/rvc_colab_out")
    if _P("/content").exists() and not _P("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
        except Exception as _exc:
            print("[drive] mount skipped:", _exc)
    try:
        _drive_dir.mkdir(parents=True, exist_ok=True)
        for p in _present:
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
    except Exception as _exc:
        print("[backup] skipped:", _exc)
    try:
        from google.colab import files as _colab_files  # type: ignore
        for p in _present:
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
    except Exception as _exc:
        print("[download] skipped (not Colab or blocked):", _exc)


---
# K3 — O15 surprisal contamination

Typicality-vs-contamination reframe. Suite sets `RUN_OPTIONAL_SCALE=False` for the first pass (fp16 primaries only).

**Flag:** `RUN_K3_O15` · **Outputs (download separately):** `O15_surprisal_contamination.csv`


<details><summary>Arm README (from standalone notebook)</summary>

# O15 — Independent surprisal contamination measure (Colab T4)

Probe 3 currently rests on a **single** proxy (Infini-gram n-gram overlap) with
**incomparable** windows across families (GSM 8, ALGO/BW 13). This notebook adds
an independent second measure: **causal LM surprisal of the problem statement**
(not the solution).

### Models (T4)
| Model | dtype | flag |
|-------|-------|------|
| `EleutherAI/pythia-2.8b` (The Pile) | fp16 | primary |
| `allenai/OLMo-2-0425-1B` (Dolma / OLMo-mix) | fp16 | primary |
| `Qwen/Qwen2.5-1.5B` | fp16 | primary |
| `EleutherAI/pythia-6.9b` | **4-bit NF4** | optional scale (`RUN_OPTIONAL_SCALE`) |
| `allenai/OLMo-2-1124-7B` | **4-bit NF4** | optional scale |

**T4 hard constraints:** fp16 only (no bf16), `attn_implementation="sdpa"`, no FlashAttention-2.

### Per problem statement (canonical **and** every bank variant)
1. **Mean per-token NLL** (negative log-likelihood).
2. Length control is applied **downstream** (`scripts/consolidate/o15_surprisal_vs_infinigram.py`): OLS residual of mean NLL on token count within family×model.
3. **Min-k%** (k=20): mean log-prob of the k% lowest-probability tokens (Carlini-style membership statistic).

### Output
`colab_out/O15_surprisal_contamination.csv` → copy to `results/raw/O15_surprisal_contamination.csv`, then run the consolidate script for residuals + Infini-gram Spearman.

**Pre-registered reading:** agreement with Infini-gram validates Probe 3; disagreement shows the field's standard proxy is unreliable. Report which.

**Secrets:** optional `HF_TOKEN` / `GITHUB_TOKEN` (Colab 🔑).

</details>


## Item queue — every bank problem statement (canonical + W*)

No Probe-1 chat wrapper: we score the **raw `problem_text`** under each base LM.
Infini-gram correlations use existing P3 scores (canonical texts); variants are
still scored here so length residuals see the full statement distribution.


In [ ]:
from __future__ import annotations
if not (RUN_K3_O15):
    print('[skip] K3_O15  (RUN_K3_O15=False)')
else:

    import csv
    import gc
    from typing import Any

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    from probes.common.clones import algo_cluster_map

    VARIANTS = ("canonical", "W1", "W2", "W3", "W4", "W5", "W6")

    # (hf_id, quant, optional_scale, corpus_note, short_name)
    MODELS: list[tuple[str, str, bool, str, str]] = [
        ("EleutherAI/pythia-2.8b", "fp16", False, "The Pile", "pythia-2.8b"),
        ("allenai/OLMo-2-0425-1B", "fp16", False, "Dolma/OLMo-mix", "olmo2-1b"),
        ("Qwen/Qwen2.5-1.5B", "fp16", False, "Qwen2.5 pretrain", "qwen2.5-1.5b"),
        ("EleutherAI/pythia-6.9b", "nf4", True, "The Pile", "pythia-6.9b"),
        ("allenai/OLMo-2-1124-7B", "nf4", True, "Dolma/OLMo-mix", "olmo2-7b"),
    ]

    O15_CSV = OUT_DIR / "O15_surprisal_contamination.csv"

    OUT_COLUMNS = [
        "family",
        "problem_id",
        "variant",
        "model",
        "model_short",
        "quantized",
        "dtype_label",
        "corpus_note",
        "n_tokens",
        "sum_nll",
        "mean_nll",
        "min_k_pct",
        "min_k_mean_logprob",
        "min_k_mean_nll",
        "n_min_k_tokens",
        "clone_family",
        "whitespace_n_tokens",
    ]


    def _norm_vt(v: str) -> str:
        v = str(v).strip()
        return "canonical" if v.lower() == "canonical" else v.upper()


    def _strip_csv_quotes(text: str) -> str:
        s = str(text)
        if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
            s = s[1:-1]
        return s


    def _load_bank(path: Path) -> pd.DataFrame:
        df = pd.read_csv(path, dtype=str).fillna("")
        df["problem_id"] = df["problem_id"].astype(str).str.strip()
        df["variant_type"] = df["variant_type"].map(_norm_vt)
        df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
        return df


    def clone_family_for(family: str, problem_id: str, cmap: dict[str, str]) -> str:
        if family == "ALGO":
            return cmap.get(problem_id, f"SINGLETON_{problem_id}")
        return f"SINGLETON_{problem_id}"


    def load_items(limit: int | None) -> list[dict[str, Any]]:
        specs = [
            ("GSM", REPO_ROOT / "data/problems/question_bank_gsm.csv"),
            ("ALGO", REPO_ROOT / "data/problems/question_bank_algo.csv"),
            ("BW", REPO_ROOT / "data/problems/question_bank_bw.csv"),
        ]
        cmap = algo_cluster_map()
        items: list[dict[str, Any]] = []
        for family, path in specs:
            df = _load_bank(path)
            for _, row in df.iterrows():
                vt = str(row["variant_type"])
                if vt not in VARIANTS:
                    continue
                text = str(row["problem_text"]).strip()
                if not text:
                    continue
                pid = str(row["problem_id"])
                items.append(
                    {
                        "family": family,
                        "problem_id": pid,
                        "variant": vt,
                        "problem_text": text,
                        "clone_family": clone_family_for(family, pid, cmap),
                        "whitespace_n_tokens": len(text.split()),
                    }
                )
        if limit is not None:
            keep: set[tuple[str, str]] = set()
            for fam in ("GSM", "ALGO", "BW"):
                ids = sorted({x["problem_id"] for x in items if x["family"] == fam})[:limit]
                keep |= {(fam, pid) for pid in ids}
            items = [x for x in items if (x["family"], x["problem_id"]) in keep]
        return items


    ITEMS = load_items(LIMIT)
    print(f"[queue] {len(ITEMS)} problem statements (LIMIT={LIMIT})")
    print(pd.DataFrame(ITEMS).groupby(["family", "variant"]).size().unstack(fill_value=0).to_string())


## Surprisal metrics on the problem statement

Causal LM forward pass on the tokenized statement. Token NLLs are for positions
`1..n-1` (token `t_i` predicted from `t_<i`). Min-k% uses the lowest-probability
`ceil(0.20 * n)` of those tokens.


In [ ]:
if not (RUN_K3_O15):
    print('[skip] K3_O15  (RUN_K3_O15=False)')
else:
    def append_rows(path: Path, rows: list[dict]) -> None:
        write_header = not path.exists() or path.stat().st_size == 0
        with path.open("a", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=OUT_COLUMNS)
            if write_header:
                w.writeheader()
            for r in rows:
                w.writerow({c: r.get(c, "") for c in OUT_COLUMNS})


    def _done_keys(path: Path) -> set[tuple[str, str, str, str]]:
        if not path.exists() or path.stat().st_size == 0:
            return set()
        df = pd.read_csv(path, dtype=str).fillna("")
        keys = set()
        for _, r in df.iterrows():
            keys.add(
                (
                    str(r["family"]),
                    str(r["problem_id"]),
                    str(r["variant"]),
                    str(r["model"]),
                )
            )
        return keys


    @torch.inference_mode()
    def statement_surprisal(
        model,
        tokenizer,
        device,
        text: str,
        *,
        min_k_pct: int = 20,
    ) -> dict[str, Any]:
        """Mean NLL + min-k% on the problem statement (no chat template, no gold)."""
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=True)
        input_ids = enc["input_ids"].to(device)
        n = int(input_ids.shape[1])
        if n < 2:
            return {
                "n_tokens": n,
                "sum_nll": float("nan"),
                "mean_nll": float("nan"),
                "min_k_mean_logprob": float("nan"),
                "min_k_mean_nll": float("nan"),
                "n_min_k_tokens": 0,
            }
        if DRY_RUN or model is None:
            # Deterministic placeholder from length (pipeline check only).
            fake = -2.0 - 0.001 * n
            return {
                "n_tokens": n,
                "sum_nll": round((-fake) * (n - 1), 6),
                "mean_nll": round(-fake, 6),
                "min_k_mean_logprob": round(fake - 0.5, 6),
                "min_k_mean_nll": round(-(fake - 0.5), 6),
                "n_min_k_tokens": max(1, int(np.ceil((n - 1) * min_k_pct / 100.0))),
            }

        out = model(input_ids=input_ids, use_cache=False)
        # logits[i] predicts token i+1 → score tokens 1..n-1
        logits = out.logits[0, :-1].float()
        targets = input_ids[0, 1:]
        log_probs = F.log_softmax(logits, dim=-1)
        tok_lp = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        tok_lp_np = tok_lp.detach().cpu().numpy()
        n_scored = int(tok_lp_np.shape[0])
        mean_lp = float(tok_lp_np.mean())
        sum_nll = float((-tok_lp_np).sum())
        mean_nll = float(-mean_lp)

        k = max(1, int(np.ceil(n_scored * float(min_k_pct) / 100.0)))
        lowest = np.sort(tok_lp_np)[:k]  # most surprising (lowest logprob)
        min_k_mean_lp = float(lowest.mean())

        del out, logits, input_ids, log_probs, tok_lp
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "n_tokens": n_scored,  # scored continuation tokens (n_input - 1)
            "sum_nll": round(sum_nll, 6),
            "mean_nll": round(mean_nll, 6),
            "min_k_mean_logprob": round(min_k_mean_lp, 6),
            "min_k_mean_nll": round(-min_k_mean_lp, 6),
            "n_min_k_tokens": k,
        }


    def load_model(model_id: str, quant: str):
        assert torch.cuda.is_available() or DRY_RUN, "GPU required (Colab T4) unless DRY_RUN."
        tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True, trust_remote_code=True)
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        if DRY_RUN:
            print(f"[model] DRY_RUN skip load: {model_id} ({quant})")
            return tok, None, torch.device("cpu")

        common = dict(
            device_map="auto",
            token=HF_TOKEN or True,
            attn_implementation="sdpa",
            torch_dtype=torch.float16,
            trust_remote_code=True,
        )
        if quant == "fp16":
            mdl = AutoModelForCausalLM.from_pretrained(model_id, **common)
            label = "fp16"
        elif quant == "nf4":
            bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id, quantization_config=bnb, **common
            )
            label = "nf4_4bit"
        else:
            raise ValueError(quant)
        mdl.eval()
        device = next(mdl.parameters()).device
        print(f"[model] {model_id}  {label}  device={device}")
        return tok, mdl, device


    def unload(mdl):
        if mdl is None:
            return
        del mdl
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    def score_item(model, tokenizer, device, item: dict, meta: dict) -> dict[str, Any]:
        m = statement_surprisal(
            model, tokenizer, device, item["problem_text"], min_k_pct=MIN_K_PCT
        )
        return {
            "family": item["family"],
            "problem_id": item["problem_id"],
            "variant": item["variant"],
            "model": meta["model_id"],
            "model_short": meta["short"],
            "quantized": str(meta["quant"] != "fp16").lower(),
            "dtype_label": meta["quant"],
            "corpus_note": meta["corpus"],
            "n_tokens": m["n_tokens"],
            "sum_nll": m["sum_nll"],
            "mean_nll": m["mean_nll"],
            "min_k_pct": MIN_K_PCT,
            "min_k_mean_logprob": m["min_k_mean_logprob"],
            "min_k_mean_nll": m["min_k_mean_nll"],
            "n_min_k_tokens": m["n_min_k_tokens"],
            "clone_family": item["clone_family"],
            "whitespace_n_tokens": item["whitespace_n_tokens"],
        }


## Run all models (resume-safe)


In [ ]:
if not (RUN_K3_O15):
    print('[skip] K3_O15  (RUN_K3_O15=False)')
else:
    active_models = [
        m for m in MODELS if (not m[2]) or RUN_OPTIONAL_SCALE
    ]
    print("[models]", [f"{s} ({q}{' optional' if opt else ''})" for _, q, opt, _, s in active_models])

    done = _done_keys(O15_CSV) if RESUME else set()
    print(f"[resume] {len(done)} rows already in {O15_CSV}")

    for model_id, quant, _opt, corpus, short in active_models:
        tok, mdl, device = load_model(model_id, quant)
        meta = {"model_id": model_id, "quant": quant, "corpus": corpus, "short": short}
        buf: list[dict] = []
        todo = [
            it for it in ITEMS
            if (it["family"], it["problem_id"], it["variant"], model_id) not in done
        ]
        print(f"[run] {short}: {len(todo)} remaining / {len(ITEMS)} total")
        for it in tqdm(todo, desc=short):
            buf.append(score_item(mdl, tok, device, it, meta))
            if len(buf) >= 32:
                append_rows(O15_CSV, buf)
                buf = []
        if buf:
            append_rows(O15_CSV, buf)
        unload(mdl)
        del tok
        gc.collect()

    print(f"\n[done] wrote {O15_CSV}")
    if O15_CSV.exists():
        out_df = pd.read_csv(O15_CSV)
        print(out_df.groupby(["model_short", "family"]).size().unstack(fill_value=0).to_string())
        print(out_df.groupby("model_short")[["mean_nll", "min_k_mean_logprob"]].mean().round(4).to_string())


### Download — K3_O15 only


In [ ]:
if not (RUN_K3_O15):
    print("[skip download] K3_O15")
else:
    from pathlib import Path as _P
    import shutil as _shutil
    _names = ['O15_surprisal_contamination.csv']
    _paths = [OUT_DIR / n for n in _names]
    _present = [p for p in _paths if p.is_file()]
    print(f"[download K3_O15] present={len(_present)}/{len(_paths)} in {OUT_DIR}")
    for p in _paths:
        print(" ", "OK" if p.is_file() else "MISSING", p.name)
    _drive_dir = _P("/content/drive/MyDrive/rvc_colab_out")
    if _P("/content").exists() and not _P("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
        except Exception as _exc:
            print("[drive] mount skipped:", _exc)
    try:
        _drive_dir.mkdir(parents=True, exist_ok=True)
        for p in _present:
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
    except Exception as _exc:
        print("[backup] skipped:", _exc)
    try:
        from google.colab import files as _colab_files  # type: ignore
        for p in _present:
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
    except Exception as _exc:
        print("[download] skipped (not Colab or blocked):", _exc)


---
# K4 — O8 mechanistic↔behavioral link

Requires `O5_teacher_forced_likelihood.csv` in `colab_out/` (from K1) and O7 PASS for GSM. Suite forces `INCLUDE_GSM=True`.

**Flag:** `RUN_K4_O8` · **Outputs (download separately):** `O8_mech_behavior_link.csv`, `O8_layer_profile.csv`, `O8_w3_binary_scores.csv`, `O8_framing.txt`


<details><summary>Arm README (from standalone notebook)</summary>

# O8 — Mechanistic↔behavioral instrument validation (replaces N3)

**N3 failed:** the outcome was constant (Llama W3 accuracy 1/60, Qwen 0/61) — you cannot correlate against a constant.

**This notebook** correlates **per-layer** canonical→W3 gold-token **rank shift** against the **continuous** O5 outcome (`delta_mean_logprob`), not binary correctness. Binary is still reported to show it is degenerate.

### Framing constraint (read before claiming anything)
This is **instrument validation only** ("does the behavioral measure track anything internal"), **never** a mechanism claim about where a split lives. arXiv 2602.04843 (Feb 2026) already did per-layer Mystery-Blocksworld mechanistic analysis.

### Models / families
| | |
|--|--|
| Models | `Qwen/Qwen2.5-1.5B-Instruct`, `Qwen/Qwen2.5-3B-Instruct` |
| Precision | **fp16 ONLY** + `attn_implementation="sdpa"` (no 4-bit/8-bit; see O6) |
| ALGO | frozen adversarial 61 (canonical + W3) |
| GSM | included **iff O7 PASS** (auto from verdict file; override with `INCLUDE_GSM`) |
| BW | **excluded** (gold-token degeneracy) |

### Outputs
- `O8_mech_behavior_link.csv` — per instance × layer
- `O8_layer_profile.csv` — Spearman(rank_shift, y) by layer with cluster-bootstrap CIs
- `O8_framing.txt` — validation-only disclaimer

Clone-family cluster bootstrap (`n_boot=5000`, seed 42), same stack as N3.

</details>


## Knobs, O7 gate, item queue (ALGO 61 + optional GSM)

`INCLUDE_GSM=None` reads O7 verdict (`PASS` → include). Set `True`/`False` to force.


In [ ]:
from __future__ import annotations
if not (RUN_K4_O8):
    print('[skip] K4_O8  (RUN_K4_O8=False)')
else:

    import csv
    import gc
    import re
    from typing import Any

    import pandas as pd
    import torch
    import torch.nn.functional as F
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer

    from probes.common.clones import algo_cluster_map
    from probes.common.cluster_inference import cluster_bootstrap_assoc
    from probes.contamination.verify import verify_gsm_answer
    from probes.contamination.verify_algo import verify_algo

    # ── knobs (override setup LIMIT/DRY_RUN/RESUME as needed) ─────────────────
    INCLUDE_GSM = True  # suite: ALGO+GSM (O7 PASS); edit master knobs to override
    N_BOOT = 5000 if not DRY_RUN else 200
    BOOT_SEED = 42
    MAX_NEW_TOKENS_GREEDY = 128  # W3 binary correctness check only

    MODELS = [
        "Qwen/Qwen2.5-1.5B-Instruct",
        "Qwen/Qwen2.5-3B-Instruct",
    ]

    PROBE1_TEMPLATE = (
        "Solve the following problem exactly and provide only the final answer "
        "in the required output format. Problem: {problem}. Format instruction: "
        "{family_specific_output_format}."
    )
    FAMILY_FORMAT = {
        "GSM": (
            "Write the final numerical answer on its own line as #### <number>. "
            "No other text after that tag."
        ),
        "ALGO": (
            "Follow the problem's required output format exactly "
            "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
        ),
    }
    FORMAT_KEYWORDS = {
        "path", "count", "selected", "coins", "scoops", "total", "answer",
        "final", "####", "#", ":", "[", "]", "{", "}", ",",
        "path:", "count:", "selected:", "coins:", "scoops:",
    }

    # Frozen ALGO adversarial pool (rebuild/FROZEN_FILTERS.md) — same as mech notebook.
    ALGO_ADV = {
        "CC": [f"CC_{i:02d}" for i in range(1, 11)],
        "SP": [
            "SP_003", "SP_004", "SP_005", "SP_019", "SP_020", "SP_021", "SP_023",
            "SP_024", "SP_026", "SP_027", "SP_028", "SP_029", "SP_030", "SP_037",
            "SP_038", "SP_039", "SP_040", "SP_042", "SP_044", "SP_045", "SP_046",
            "SP_047", "SP_048", "SP_062", "SP_063", "SP_064", "SP_065", "SP_066",
            "SP_068", "SP_069", "SP_070", "SP_071", "SP_072", "SP_073",
        ],
        "WIS": [
            "WIS_003", "WIS_004", "WIS_013", "WIS_014", "WIS_015", "WIS_016",
            "WIS_017", "WIS_018", "WIS_019", "WIS_020", "WIS_023", "WIS_024",
            "WIS_025", "WIS_026", "WIS_027", "WIS_028", "WIS_029",
        ],
    }
    ALGO_ADV_IDS = ALGO_ADV["CC"] + ALGO_ADV["SP"] + ALGO_ADV["WIS"]
    assert len(ALGO_ADV_IDS) == 61

    O5_CANDIDATES = [
        OUT_DIR / "O5_teacher_forced_likelihood.csv",
        REPO_ROOT / "results/raw/O5_teacher_forced_likelihood.csv",
        Path("/content/drive/MyDrive/rvc_colab_out/O5_teacher_forced_likelihood.csv"),
    ]
    O7_VERDICT_CANDIDATES = [
        OUT_DIR / "O7_gsm_degeneracy_verdict.txt",
        REPO_ROOT / "results/derived/O7_gsm_degeneracy_verdict.txt",
        OUT_DIR / "O7_gsm_degeneracy_check.csv",
        REPO_ROOT / "results/derived/O7_gsm_degeneracy_check.csv",
        Path("/content/drive/MyDrive/rvc_colab_out/O7_gsm_degeneracy_verdict.txt"),
    ]

    O8_LINK = OUT_DIR / "O8_mech_behavior_link.csv"
    O8_PROFILE = OUT_DIR / "O8_layer_profile.csv"
    O8_FRAMING = OUT_DIR / "O8_framing.txt"
    O8_BINARY = OUT_DIR / "O8_w3_binary_scores.csv"  # per-item greedy W3 correctness

    LINK_COLUMNS = [
        "family", "model", "problem_id", "layer", "n_layers",
        "rank_canonical", "rank_w3", "rank_shift_canonical_minus_w3",
        "mean_logprob_canonical", "mean_logprob_w3", "delta_mean_logprob",
        "w3_correct", "binary_degenerate_cell", "clone_family",
        "gold_content_canonical", "gold_content_w3",
        "gold_token_id_canonical", "gold_token_id_w3",
        "gold_token_decoded_canonical", "gold_token_decoded_w3",
        "framing",
    ]
    PROFILE_COLUMNS = [
        "family", "model", "layer", "y",
        "n", "n_clusters",
        "spearman_rho", "ci_low", "ci_high", "p_value",
        "p_value_method", "bootstrap", "n_boot", "seed",
        "y_nunique", "binary_outcome_degenerate", "note", "framing",
    ]
    FRAMING = (
        "Instrument validation only — does the behavioral measure track anything "
        "internal. Not a mechanism claim. See arXiv 2602.04843."
    )


    def _norm_vt(v: str) -> str:
        v = str(v).strip()
        return "canonical" if v.lower() == "canonical" else v.upper()


    def _strip_csv_quotes(text: str) -> str:
        s = str(text)
        if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
            s = s[1:-1]
        return s


    def _norm_bank(path: Path) -> pd.DataFrame:
        df = pd.read_csv(path, dtype=str).fillna("")
        df["problem_id"] = df["problem_id"].astype(str).str.strip()
        df["variant_type"] = df["variant_type"].map(_norm_vt)
        df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
        df["correct_answer"] = df["correct_answer"].map(_strip_csv_quotes)
        return df


    def resolve_o7_include_gsm() -> bool:
        if INCLUDE_GSM is not None:
            print(f"[o7] INCLUDE_GSM forced={INCLUDE_GSM}")
            return bool(INCLUDE_GSM)
        for p in O7_VERDICT_CANDIDATES:
            if not p.exists():
                continue
            if p.suffix == ".txt":
                text = p.read_text().upper()
                if "VERDICT: PASS" in text or "\nPASS" in text or text.strip().startswith("PASS"):
                    print(f"[o7] PASS from {p}")
                    return True
                if "VERDICT: FAIL" in text or "FAIL" in text.split("VERDICT:", 1)[-1][:20]:
                    print(f"[o7] FAIL from {p}")
                    return False
            if p.suffix == ".csv":
                df = pd.read_csv(p, dtype=str)
                if "verdict" in df.columns and (df["verdict"].str.upper() == "FAIL").any():
                    print(f"[o7] FAIL from {p}")
                    return False
                if "verdict" in df.columns and (df["verdict"].str.upper() == "PASS").all():
                    print(f"[o7] PASS from {p}")
                    return True
        # Local preview / O7 not uploaded yet: default PASS (GSM content screen passed in O7 dry-run).
        print("[o7] verdict file not found — defaulting INCLUDE_GSM=True (override if O7 FAIL)")
        return True


    def gsm_gold_content(correct_answer: str) -> str:
        s = str(correct_answer).strip()
        s = re.sub(r"^####\s*", "", s).replace(",", "")
        try:
            f = float(s)
            return str(int(f)) if f == int(f) else str(f)
        except ValueError:
            m = re.findall(r"-?\d+(?:\.\d+)?", s)
            if not m:
                raise ValueError(f"no numeric gold in {correct_answer!r}")
            return m[-1]


    def algo_gold_content(problem_id: str, correct_answer: str) -> str:
        s = str(correct_answer)
        pid = str(problem_id).strip().upper()
        if pid.startswith("SP"):
            m = re.search(r"Cost\s*:\s*(-?\d+)", s, flags=re.I)
            if not m:
                raise ValueError(f"{problem_id}: no Cost: gold")
            return m.group(1)
        if pid.startswith("CC"):
            m = re.search(r"(?:Count|Total)\s*:\s*(-?\d+)", s, flags=re.I)
            if not m:
                raise ValueError(f"{problem_id}: no Count:/Total: gold")
            return m.group(1)
        if pid.startswith("WIS"):
            m = re.search(r"Total\s*:\s*(-?\d+)", s, flags=re.I)
            if not m:
                raise ValueError(f"{problem_id}: no Total: gold")
            return m.group(1)
        raise ValueError(f"{problem_id}: unknown ALGO subtype")


    def build_user(problem_text: str, family: str) -> str:
        return PROBE1_TEMPLATE.format(
            problem=str(problem_text).strip(),
            family_specific_output_format=FAMILY_FORMAT[family],
        )


    def bank_row(df: pd.DataFrame, pid: str, vt: str) -> pd.Series:
        sub = df[(df.problem_id == pid) & (df.variant_type == vt)]
        if sub.empty:
            raise KeyError(f"{pid}/{vt}")
        return sub.iloc[0]


    algo_df = _norm_bank(REPO_ROOT / "data/problems/question_bank_algo.csv")
    gsm_df = _norm_bank(REPO_ROOT / "data/problems/question_bank_gsm.csv")
    paired_algo = sorted(
        set(algo_df.loc[algo_df.variant_type == "canonical", "problem_id"])
        & set(algo_df.loc[algo_df.variant_type == "W3", "problem_id"])
    )
    ALGO_IDS = [pid for pid in ALGO_ADV_IDS if pid in set(paired_algo)]
    assert len(ALGO_IDS) == 61, len(ALGO_IDS)

    include_gsm = resolve_o7_include_gsm()
    GSM_IDS = sorted(
        set(gsm_df.loc[gsm_df.variant_type == "canonical", "problem_id"])
        & set(gsm_df.loc[gsm_df.variant_type == "W3", "problem_id"])
    ) if include_gsm else []

    cmap = algo_cluster_map()


    def clone_for(family: str, pid: str) -> str:
        if family == "ALGO":
            return cmap.get(pid, f"SINGLETON_{pid}")
        return f"SINGLETON_{pid}"


    def make_item(family: str, pid: str, vt: str, df: pd.DataFrame) -> dict[str, Any]:
        r = bank_row(df, pid, vt)
        if family == "ALGO":
            gold = algo_gold_content(pid, r["correct_answer"])
        else:
            gold = gsm_gold_content(r["correct_answer"])
        return {
            "family": family,
            "problem_id": pid,
            "variant": vt,
            "problem_text": str(r["problem_text"]),
            "correct_answer": str(r["correct_answer"]),
            "gold_content": gold,
            "problem_subtype": str(r.get("problem_subtype", "")).strip().lower(),
            "difficulty_params": str(r.get("difficulty_params", "{}") or "{}"),
            "clone_family": clone_for(family, pid),
        }


    ITEMS: list[dict[str, Any]] = []
    for pid in ALGO_IDS:
        for vt in ("canonical", "W3"):
            ITEMS.append(make_item("ALGO", pid, vt, algo_df))
    for pid in GSM_IDS:
        for vt in ("canonical", "W3"):
            ITEMS.append(make_item("GSM", pid, vt, gsm_df))

    if LIMIT is not None:
        keep_algo = set(ALGO_IDS[:LIMIT])
        keep_gsm = set(GSM_IDS[:LIMIT])
        ITEMS = [
            x for x in ITEMS
            if (x["family"] == "ALGO" and x["problem_id"] in keep_algo)
            or (x["family"] == "GSM" and x["problem_id"] in keep_gsm)
        ]

    n_algo = len({x["problem_id"] for x in ITEMS if x["family"] == "ALGO"})
    n_gsm = len({x["problem_id"] for x in ITEMS if x["family"] == "GSM"})
    print(f"[queue] {len(ITEMS)} prompts  ALGO_ids={n_algo}  GSM_ids={n_gsm}  include_gsm={include_gsm}")
    print(pd.DataFrame(ITEMS).groupby(["family", "variant"]).size().unstack(fill_value=0).to_string())
    assert not any(x["family"] == "BW" for x in ITEMS), "BW must stay excluded"
    O8_FRAMING.write_text(FRAMING + "\n")


## Logit-lens readout + O5 mean_logprob + W3 greedy binary

Per-layer gold-token rank at the last prompt position (same targeting as the mechanistic notebook).  
`delta_mean_logprob = mean_logprob_canonical − mean_logprob_w3` (parallel to `rank_shift_canonical_minus_w3`).  
W3 binary correctness via greedy decode + released verifiers (to demonstrate degeneracy).


In [ ]:
if not (RUN_K4_O8):
    print('[skip] K4_O8  (RUN_K4_O8=False)')
else:
    def wrap_chat(tokenizer, user_text: str) -> str:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_text}],
            add_generation_prompt=True,
            tokenize=False,
        )


    def resolve_target_token(tokenizer, prompt: str, answer: str) -> tuple[int, str, list[int], str]:
        def enc(text: str) -> list[int]:
            return tokenizer.encode(text, add_special_tokens=False)

        prompt_ids = enc(prompt)
        answer_ids_bare = enc(answer)
        candidates: list[tuple[str, int, int, list[int]]] = []
        for sep in ("", " "):
            joint = enc(prompt + sep + answer)
            if len(joint) <= len(prompt_ids):
                continue
            if joint[: len(prompt_ids)] != prompt_ids:
                continue
            rest = joint[len(prompt_ids) :]
            candidates.append((sep, int(rest[0]), len(joint), rest))
        if not candidates:
            if not answer_ids_bare:
                return -1, "", [], "EMPTY"
            tid = int(answer_ids_bare[0])
            return tid, tokenizer.decode([tid]), answer_ids_bare, "FALLBACK"
        candidates.sort(key=lambda c: c[2])
        sep, tid, _, rest = candidates[0]
        return tid, tokenizer.decode([tid]), rest, repr(sep)


    def assert_content_gold(decoded: str, family: str) -> None:
        d = decoded.strip().lower()
        compact = d.replace(" ", "")
        if compact in FORMAT_KEYWORDS or d in FORMAT_KEYWORDS:
            raise AssertionError(f"format-keyword gold token: {decoded!r}")
        if family in {"GSM", "ALGO"} and not re.search(r"\d", decoded):
            raise AssertionError(f"{family} gold token must contain a digit: {decoded!r}")


    def resolve_continuation(tokenizer, prompt: str, answer: str) -> tuple[list[int], list[int]]:
        def enc(text: str) -> list[int]:
            return tokenizer.encode(text, add_special_tokens=False)

        prompt_ids = enc(prompt)
        for sep in ("", " "):
            joint = enc(prompt + sep + str(answer))
            if len(joint) > len(prompt_ids) and joint[: len(prompt_ids)] == prompt_ids:
                return prompt_ids, joint[len(prompt_ids) :]
        return prompt_ids, enc(str(answer))


    @torch.inference_mode()
    def readout_layers(model, tokenizer, device, user_text: str, gold_content: str, family: str) -> dict:
        prompt = wrap_chat(tokenizer, user_text)
        tid, decoded, gold_ids, sep_note = resolve_target_token(tokenizer, prompt, gold_content)
        assert_content_gold(decoded, family)
        if DRY_RUN or model is None:
            n = 8 if DRY_RUN else int(getattr(model.config, "num_hidden_layers", 8))
            # Inject item-hash variation so continuous correlations are defined in smoke tests.
            h = abs(hash(gold_content + user_text[:40])) % 97
            ranks = [1 + ((h + i) % 50) for i in range(n)]
            return {
                "gold_token_id": tid if tid >= 0 else 1,
                "gold_token_decoded": decoded or "1",
                "sep_note": "DRY_RUN",
                "n_layers": n,
                "ranks": ranks,
                "mean_logprob": round(-0.2 - 0.01 * (h % 10), 6),
            }

        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        out = model(**inputs, output_hidden_states=True, use_cache=False)
        hidden_states = out.hidden_states[1:]  # layer 0 = first block
        W_U = model.lm_head.weight.detach().float()
        ranks = []
        for layer_h in hidden_states:
            h = layer_h[0, -1, :].float()
            logits = h @ W_U.T
            target_logit = logits[tid]
            rank = int((logits > target_logit).sum().item()) + 1
            ranks.append(rank)
        del out, hidden_states, inputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Teacher-forced mean_logprob of full gold content (O5 primary metric; length-normalized).
        prompt_ids, gold_toks = resolve_continuation(tokenizer, prompt, gold_content)
        n_prompt, n_gold = len(prompt_ids), len(gold_toks)
        if n_gold == 0:
            mean_lp = float("nan")
        else:
            inp = torch.tensor([prompt_ids + gold_toks], dtype=torch.long, device=device)
            out2 = model(input_ids=inp, use_cache=False)
            glo = out2.logits[0, n_prompt - 1 : n_prompt + n_gold - 1].float()
            log_probs = F.log_softmax(glo, dim=-1)
            gt = torch.tensor(gold_toks, device=device, dtype=torch.long)
            sum_lp = float(log_probs.gather(1, gt.unsqueeze(1)).squeeze(1).sum().item())
            mean_lp = sum_lp / n_gold
            del out2, inp
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return {
            "gold_token_id": int(tid),
            "gold_token_decoded": decoded,
            "sep_note": sep_note,
            "n_layers": len(ranks),
            "ranks": ranks,
            "mean_logprob": round(float(mean_lp), 6),
        }


    @torch.inference_mode()
    def greedy_w3_correct(model, tokenizer, device, item: dict) -> bool:
        """Binary W3 correctness for degeneracy contrast (not the primary outcome)."""
        if item["variant"] != "W3":
            raise ValueError("greedy_w3_correct expects W3 items")
        user = build_user(item["problem_text"], item["family"])
        prompt = wrap_chat(tokenizer, user)
        if DRY_RUN or model is None:
            # Near-constant False → binary degeneracy in smoke (flip ~1/40).
            return (abs(hash(item["problem_id"])) % 40) == 0
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        gen = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS_GREEDY,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        new_tokens = gen[0, inputs["input_ids"].shape[1] :]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        del gen, inputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if item["family"] == "GSM":
            return bool(verify_gsm_answer(text, item["correct_answer"]))
        ok, _reason, _meta = verify_algo(
            item["problem_id"],
            text,
            item["correct_answer"],
            item["problem_subtype"],
            "W3",
            item["difficulty_params"],
        )
        return bool(ok)


    def load_o5_lookup() -> dict[tuple[str, str, str, str], float]:
        """(model, family, problem_id, variant) → mean_logprob from O5 CSV if present."""
        path = next((p for p in O5_CANDIDATES if p.exists()), None)
        if path is None:
            print("[o5] no O5 CSV found — will compute mean_logprob in-session (identical teacher-force)")
            return {}
        df = pd.read_csv(path, dtype=str)
        need = {"model", "family", "problem_id", "variant", "mean_logprob"}
        if not need.issubset(df.columns):
            print(f"[o5] {path} missing columns — ignoring")
            return {}
        out: dict[tuple[str, str, str, str], float] = {}
        for r in df.itertuples(index=False):
            try:
                out[(str(r.model), str(r.family), str(r.problem_id), str(r.variant))] = float(r.mean_logprob)
            except Exception:
                continue
        print(f"[o5] loaded {len(out)} mean_logprob cells from {path}")
        return out


    def load_model_fp16(model_id: str):
        assert torch.cuda.is_available() or DRY_RUN, "GPU required unless DRY_RUN"
        tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True)
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        if DRY_RUN:
            print(f"[model] DRY_RUN skip weights: {model_id} fp16")
            return tok, None, torch.device("cpu")
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            attn_implementation="sdpa",
            token=HF_TOKEN or True,
        )
        mdl.eval()
        device = next(mdl.parameters()).device
        print(f"[model] {model_id}  fp16 + sdpa  layers={mdl.config.num_hidden_layers}  device={device}")
        return tok, mdl, device


    def unload(mdl):
        if mdl is None:
            return
        del mdl
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## Run readouts → instance×layer CSV → per-layer Spearman profile

Primary y = `delta_mean_logprob` (continuous, from O5 / in-session).  
Secondary y = `w3_correct` (binary) — expect `binary_outcome_degenerate=True` when y is constant.


In [ ]:
if not (RUN_K4_O8):
    print('[skip] K4_O8  (RUN_K4_O8=False)')
else:
    o5_lookup = load_o5_lookup()


    def append_link_rows(rows: list[dict[str, Any]]) -> None:
        if not rows:
            return
        write_header = not O8_LINK.exists()
        with O8_LINK.open("a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=LINK_COLUMNS, extrasaction="ignore")
            if write_header:
                w.writeheader()
            for r in rows:
                w.writerow({k: r.get(k, "") for k in LINK_COLUMNS})


    def done_pairs(path: Path) -> set[tuple[str, str, str]]:
        """Completed (model, family, problem_id) pairs (both variants written)."""
        if not path.exists():
            return set()
        df = pd.read_csv(path, dtype=str)
        if not {"model", "family", "problem_id", "layer"}.issubset(df.columns):
            return set()
        # A pair is done if layer 0 exists (implies full layer stack written together).
        sub = df[pd.to_numeric(df["layer"], errors="coerce") == 0]
        return set(zip(sub["model"], sub["family"], sub["problem_id"]))


    done = done_pairs(O8_LINK) if RESUME else set()
    print(f"[resume] {len(done)} instance keys in {O8_LINK}")

    binary_rows: list[dict[str, Any]] = []

    for model_id in MODELS:
        # Unique problem keys still pending for this model
        keys = sorted({(it["family"], it["problem_id"]) for it in ITEMS})
        pending_keys = [(f, p) for f, p in keys if (model_id, f, p) not in done]
        print(f"\n=== {model_id}  pending_instances={len(pending_keys)}/{len(keys)} ===")
        if not pending_keys:
            continue
        tok, mdl, device = load_model_fp16(model_id)
        by_key: dict[tuple[str, str], dict[str, dict]] = {}
        for it in ITEMS:
            by_key.setdefault((it["family"], it["problem_id"]), {})[it["variant"]] = it

        try:
            for fam, pid in tqdm(pending_keys, desc=model_id.split("/")[-1]):
                can_it = by_key[(fam, pid)]["canonical"]
                w3_it = by_key[(fam, pid)]["W3"]
                can_user = build_user(can_it["problem_text"], fam)
                w3_user = build_user(w3_it["problem_text"], fam)
                can_m = readout_layers(mdl, tok, device, can_user, can_it["gold_content"], fam)
                w3_m = readout_layers(mdl, tok, device, w3_user, w3_it["gold_content"], fam)

                # Prefer O5 CSV mean_logprob when present for this model cell.
                mlp_can = o5_lookup.get((model_id, fam, pid, "canonical"), can_m["mean_logprob"])
                mlp_w3 = o5_lookup.get((model_id, fam, pid, "W3"), w3_m["mean_logprob"])
                delta = float(mlp_can) - float(mlp_w3)

                w3_ok = greedy_w3_correct(mdl, tok, device, w3_it)
                binary_rows.append(
                    {
                        "family": fam,
                        "model": model_id,
                        "problem_id": pid,
                        "w3_correct": bool(w3_ok),
                        "clone_family": can_it["clone_family"],
                    }
                )

                n_layers = min(can_m["n_layers"], w3_m["n_layers"])
                buf = []
                for layer in range(n_layers):
                    rc = int(can_m["ranks"][layer])
                    rw = int(w3_m["ranks"][layer])
                    buf.append(
                        {
                            "family": fam,
                            "model": model_id,
                            "problem_id": pid,
                            "layer": layer,
                            "n_layers": n_layers,
                            "rank_canonical": rc,
                            "rank_w3": rw,
                            "rank_shift_canonical_minus_w3": rc - rw,
                            "mean_logprob_canonical": mlp_can,
                            "mean_logprob_w3": mlp_w3,
                            "delta_mean_logprob": round(delta, 6),
                            "w3_correct": bool(w3_ok),
                            "binary_degenerate_cell": "",  # filled in profile pass
                            "clone_family": can_it["clone_family"],
                            "gold_content_canonical": can_it["gold_content"],
                            "gold_content_w3": w3_it["gold_content"],
                            "gold_token_id_canonical": can_m["gold_token_id"],
                            "gold_token_id_w3": w3_m["gold_token_id"],
                            "gold_token_decoded_canonical": can_m["gold_token_decoded"],
                            "gold_token_decoded_w3": w3_m["gold_token_decoded"],
                            "framing": FRAMING,
                        }
                    )
                append_link_rows(buf)
        finally:
            unload(mdl)

    if binary_rows:
        pd.DataFrame(binary_rows).drop_duplicates(
            ["family", "model", "problem_id"], keep="last"
        ).to_csv(O8_BINARY, index=False)

    link = pd.read_csv(O8_LINK)
    print(f"[link] rows={len(link)}  models={sorted(link.model.unique())}")


    def profile_block(sub: pd.DataFrame, family: str, model_id: str, layer: int, y_col: str) -> dict:
        x = sub["rank_shift_canonical_minus_w3"]
        y = sub[y_col]
        if y_col == "w3_correct":
            y = y.astype(str).str.lower().isin({"true", "1", "yes"}).astype(int)
        else:
            y = pd.to_numeric(y, errors="coerce")
        x = pd.to_numeric(x, errors="coerce")
        mask = x.notna() & y.notna()
        x, y = x[mask], y[mask]
        clusters = sub.loc[mask, "clone_family"].astype(str).tolist()
        y_nunique = int(pd.Series(y).nunique(dropna=True))
        binary_degen = y_col == "w3_correct" and y_nunique < 2
        note = FRAMING
        if binary_degen:
            note = (
                f"BINARY DEGENERATE: {y_col} is constant "
                f"(nunique={y_nunique}, n={len(y)}, mean={float(y.mean()) if len(y) else float('nan'):.4f}). "
                "Spearman undefined — this is why N3 collapsed; use delta_mean_logprob."
            )
            return {
                "family": family,
                "model": model_id,
                "layer": layer,
                "y": y_col,
                "n": int(len(y)),
                "n_clusters": len(set(clusters)),
                "spearman_rho": "",
                "ci_low": "",
                "ci_high": "",
                "p_value": "",
                "p_value_method": "cluster_bootstrap_two_sided",
                "bootstrap": "cluster_by_clone_family",
                "n_boot": N_BOOT,
                "seed": BOOT_SEED,
                "y_nunique": y_nunique,
                "binary_outcome_degenerate": True,
                "note": note,
                "framing": FRAMING,
            }
        if int(pd.Series(x).nunique(dropna=True)) < 2:
            return {
                "family": family,
                "model": model_id,
                "layer": layer,
                "y": y_col,
                "n": int(len(y)),
                "n_clusters": len(set(clusters)),
                "spearman_rho": "",
                "ci_low": "",
                "ci_high": "",
                "p_value": "",
                "p_value_method": "cluster_bootstrap_two_sided",
                "bootstrap": "cluster_by_clone_family",
                "n_boot": N_BOOT,
                "seed": BOOT_SEED,
                "y_nunique": y_nunique,
                "binary_outcome_degenerate": False,
                "note": "rank_shift constant — correlation undefined; " + FRAMING,
                "framing": FRAMING,
            }
        res = cluster_bootstrap_assoc(
            x, y, clusters, kind="spearman", n_boot=N_BOOT, seed=BOOT_SEED
        )

        def _r(v):
            return round(float(v), 4) if v == v else ""

        return {
            "family": family,
            "model": model_id,
            "layer": layer,
            "y": y_col,
            "n": res["n"],
            "n_clusters": res["n_clusters"],
            "spearman_rho": _r(res["estimate"]),
            "ci_low": _r(res["ci_low"]),
            "ci_high": _r(res["ci_high"]),
            "p_value": _r(res["p_clustered"]),
            "p_value_method": "cluster_bootstrap_two_sided",
            "bootstrap": "cluster_by_clone_family",
            "n_boot": N_BOOT,
            "seed": BOOT_SEED,
            "y_nunique": y_nunique,
            "binary_outcome_degenerate": False,
            "note": note,
            "framing": FRAMING,
        }


    profile_rows: list[dict] = []
    for (fam, model_id), g in link.groupby(["family", "model"]):
        # Mark binary degeneracy at the instance level for this model×family
        inst = g.drop_duplicates(["problem_id"])
        ybin = inst["w3_correct"].astype(str).str.lower().isin({"true", "1", "yes"})
        bin_degen = int(ybin.nunique()) < 2
        if bin_degen:
            print(
                f"[binary] DEGENERATE  {model_id} / {fam}: "
                f"w3_correct nunique={ybin.nunique()}  "
                f"acc={float(ybin.mean()):.4f}  n={len(ybin)}"
            )
        else:
            print(
                f"[binary] ok variation  {model_id} / {fam}: "
                f"acc={float(ybin.mean()):.4f}  n={len(ybin)}"
            )
        for layer, gl in g.groupby(pd.to_numeric(g["layer"], errors="coerce")):
            if pd.isna(layer):
                continue
            layer_i = int(layer)
            for y_col in ("delta_mean_logprob", "w3_correct"):
                profile_rows.append(profile_block(gl, fam, model_id, layer_i, y_col))

    prof = pd.DataFrame(profile_rows, columns=PROFILE_COLUMNS)
    prof.to_csv(O8_PROFILE, index=False)

    print("\n=== O8_layer_profile.csv (final layer only, preview) ===")
    final_layers = (
        link.groupby(["family", "model"])["n_layers"].first().astype(int) - 1
    )
    preview = []
    for (fam, model_id), g in prof.groupby(["family", "model"]):
        fl = int(final_layers.get((fam, model_id), g["layer"].max()))
        preview.append(g[g["layer"] == fl])
    if preview:
        print(pd.concat(preview).to_string(index=False))

    print(f"\n[wrote] {O8_LINK} ({len(link)} rows)")
    print(f"[wrote] {O8_PROFILE} ({len(prof)} rows)")
    print(f"[wrote] {O8_FRAMING}")
    print("\n" + FRAMING)


### Download — K4_O8 only


In [ ]:
if not (RUN_K4_O8):
    print("[skip download] K4_O8")
else:
    from pathlib import Path as _P
    import shutil as _shutil
    _names = ['O8_mech_behavior_link.csv', 'O8_layer_profile.csv', 'O8_w3_binary_scores.csv', 'O8_framing.txt']
    _paths = [OUT_DIR / n for n in _names]
    _present = [p for p in _paths if p.is_file()]
    print(f"[download K4_O8] present={len(_present)}/{len(_paths)} in {OUT_DIR}")
    for p in _paths:
        print(" ", "OK" if p.is_file() else "MISSING", p.name)
    _drive_dir = _P("/content/drive/MyDrive/rvc_colab_out")
    if _P("/content").exists() and not _P("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
        except Exception as _exc:
            print("[drive] mount skipped:", _exc)
    try:
        _drive_dir.mkdir(parents=True, exist_ok=True)
        for p in _present:
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
    except Exception as _exc:
        print("[backup] skipped:", _exc)
    try:
        from google.colab import files as _colab_files  # type: ignore
        for p in _present:
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
    except Exception as _exc:
        print("[download] skipped (not Colab or blocked):", _exc)


---
# K6 — O14b naming likelihood

Controlled BW naming bank; Qwen 1.5B/3B teacher-forced + greedy.

**Flag:** `RUN_K6_O14B` · **Outputs (download separately):** `O14b_naming_likelihood.csv`, `O14b_naming_analysis.csv`


<details><summary>Arm README (from standalone notebook)</summary>

# O14b — Controlled Blocksworld naming intervention (teacher-forced likelihood)

**Colab T4 · GPU required · no OpenRouter.**

The API O14 eval is blocked (`OPENROUTER_API_KEY` 401). This notebook runs the
**same controlled n=120 × 3-arm bank** using **teacher-forced gold-plan
`mean_logprob`**, which remains informative even when open models are at floor
on Blocksworld accuracy.

### Bank
`results/derived/O14_naming_bank.jsonl` — 120 pairs × 3 arms = **360 rows**.

| Arm | Naming |
|-----|--------|
| `A_sequential` | `a, b, c, …` |
| `B_scattered` | scattered letters |
| `C_indexed` | `b1, b2, …` |

Byte-level `assert_only_block_ids_differ` was already validated when the bank was built.

### Models (T4 hard constraints)
| Model | load |
|-------|------|
| `Qwen/Qwen2.5-1.5B-Instruct` | fp16, `attn_implementation="sdpa"` |
| `Qwen/Qwen2.5-3B-Instruct` | fp16, `attn_implementation="sdpa"` |

**Reuse O5 teacher-forcing exactly** (`wrap_chat` / `resolve_continuation` /
`teacher_forced_metrics` / Appendix-N `PROBE1_TEMPLATE` + BW `FAMILY_FORMAT`).
Do not reimplement.

### Outputs
- `colab_out/O14b_naming_likelihood.csv` → `results/raw/O14b_naming_likelihood.csv`
- `colab_out/O14b_naming_analysis.csv` → `results/derived/O14b_naming_analysis.csv`

---

## SCOPE CAVEAT (notebook + paper)

This measures **belief shift (gold-plan likelihood) in small open models**, **not**
accuracy in the six frontier models where L1 was observed. It is a controlled
**n=120** intervention replacing an **n=13** observational stratification — a
genuine upgrade — but **it is not the same experiment**.

---

## PRE-REGISTERED INTERPRETATION (do not revise after seeing results)

Write this **before** the run. Both outcomes are reported.

1. **Substantial naming effect on `mean_logprob`** (paired arm contrast; 95% CI
   for Δ excludes 0 under pair_id cluster bootstrap) → block naming is a real
   **lexical** variable; L1 is reframed as partly lexical; this becomes a
   **primary finding**.
2. **Null-compatible** (CI includes 0 on primary A−B contrast for both models) →
   L1 **survives** the naming confound with a controlled n=120 on open models,
   with the explicit caveat that this does **not** test the frontier models
   where L1 was observed.

Greedy accuracy is recorded for completeness. If it is at floor, flag
`BINARY_DEGENERATE` so nobody mistakes a floor for a null on the likelihood
endpoint.

</details>


In [ ]:
if not (RUN_K6_O14B):
    print('[skip] K6_O14B  (RUN_K6_O14B=False)')
else:
    MAX_NEW_TOKENS = int(MAX_NEW_TOKENS_BW)
    print('[K6] MAX_NEW_TOKENS', MAX_NEW_TOKENS, 'LIMIT_PAIRS', LIMIT_PAIRS)


## Pre-registration gate (must print before any scoring)

Confirm the interpretation and scope caveat are frozen. Edit nothing below after
this cell has run on real data.


In [ ]:
if not (RUN_K6_O14B):
    print('[skip] K6_O14B  (RUN_K6_O14B=False)')
else:
    PREREG = {
        "endpoint": "mean_logprob of gold plan under Probe-1 prompt (O5 path)",
        "primary_contrast": "A_sequential − B_scattered (paired within pair_id)",
        "secondary_contrasts": ["A_sequential − C_indexed", "B_scattered − C_indexed"],
        "inference": "cluster bootstrap on pair_id, B=5000, seed=42; CI excludes 0",
        "substantial_naming_effect": (
            "primary Δ mean_logprob 95% CI excludes 0 → primary finding / L1 partly lexical"
        ),
        "null_compatible": (
            "primary CI includes 0 for both models → L1 survives naming confound on open "
            "models; does NOT test frontier L1 models"
        ),
        "scope_caveat": (
            "belief shift in Qwen-1.5B/3B, not frontier accuracy; controlled n=120 "
            "replaces n=13 observational stratification — upgrade, not the same experiment"
        ),
        "greedy": "record accuracy; BINARY_DEGENERATE if mean acc ≤ 0.05",
    }
    for k, v in PREREG.items():
        print(f"{k}:\n  {v}\n")
    assert N_BOOT == 5000 and SEED == 42
    print("[prereg] frozen — proceed to score")


## Load O14 naming bank (360 rows)


In [ ]:
from __future__ import annotations
if not (RUN_K6_O14B):
    print('[skip] K6_O14B  (RUN_K6_O14B=False)')
else:

    import csv
    import gc
    import json
    from pathlib import Path
    from typing import Any

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer

    from probes.contamination.verify import verify_answer

    BANK_CANDIDATES = [
        REPO_ROOT / "results/derived/O14_naming_bank.jsonl",
        Path("/content/drive/MyDrive/rvc_colab_out/O14_naming_bank.jsonl"),
    ]
    BANK_PATH = next((p for p in BANK_CANDIDATES if p.is_file()), None)
    assert BANK_PATH is not None, f"Missing O14 bank; tried {BANK_CANDIDATES}"

    rows_bank: list[dict[str, Any]] = []
    with BANK_PATH.open() as f:
        for line in f:
            line = line.strip()
            if line:
                rows_bank.append(json.loads(line))

    bank_df = pd.DataFrame(rows_bank)
    assert len(bank_df) == 360, f"expected 360 rows, got {len(bank_df)}"
    assert bank_df["pair_id"].nunique() == 120
    assert set(bank_df["arm"]) == {"A_sequential", "B_scattered", "C_indexed"}

    if LIMIT_PAIRS is not None:
        keep = sorted(bank_df["pair_id"].unique())[: int(LIMIT_PAIRS)]
        bank_df = bank_df[bank_df["pair_id"].isin(keep)].copy()
        print(f"[smoke] LIMIT_PAIRS={LIMIT_PAIRS} → {len(bank_df)} rows")

    print(f"[bank] {BANK_PATH}")
    print(bank_df.groupby("arm").size().to_string())
    print("num_blocks dist:\n", bank_df.drop_duplicates("pair_id")["num_blocks"].value_counts().sort_index().to_string())


## O5 teacher-forcing primitives (copied verbatim — do not edit)

Appendix-N Probe-1 prompt + chat wrap + joint-encode gold continuation.
Identical to `colab/o5_teacher_forced_likelihood.ipynb` / `_build_notebooks.py` NB3.


In [ ]:
if not (RUN_K6_O14B):
    print('[skip] K6_O14B  (RUN_K6_O14B=False)')
else:
    # Appendix N Probe-1 template (paper/appendix.tex \label{app:prompts})
    PROBE1_TEMPLATE = (
        "Solve the following problem exactly and provide only the final answer "
        "in the required output format. Problem: {problem}. Format instruction: "
        "{family_specific_output_format}."
    )

    FAMILY_FORMAT = {
        "GSM": (
            "Write the final numerical answer on its own line as #### <number>. "
            "No other text after that tag."
        ),
        "ALGO": (
            "Follow the problem's required output format exactly "
            "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
        ),
        "BW": (
            "A numbered list of actions only. Each action must be one of the "
            "permitted operators with their arguments. No explanation."
        ),
    }

    MODELS: list[tuple[str, str]] = [
        ("Qwen/Qwen2.5-1.5B-Instruct", "fp16"),
        ("Qwen/Qwen2.5-3B-Instruct", "fp16"),
    ]

    O14B_CSV = OUT_DIR / "O14b_naming_likelihood.csv"
    O14B_ANALYSIS = OUT_DIR / "O14b_naming_analysis.csv"

    COLUMNS = [
        "pair_id",
        "arm",
        "problem_id",
        "model",
        "model_short",
        "num_blocks",
        "plan_length",
        "n_gold_tokens",
        "sum_logprob",
        "mean_logprob",
        "gold_first_token_rank",
        "gold_first_token_logprob",
        "prompt_n_tokens",
        "sep_note",
        "greedy_response",
        "greedy_correct",
        "binary_flag",
    ]


    def build_prompt(problem_text: str, family: str) -> str:
        """Identical Probe-1 user string construction as the behavioural Colab notebooks."""
        return PROBE1_TEMPLATE.format(
            problem=problem_text.strip(),
            family_specific_output_format=FAMILY_FORMAT[family],
        )


    def wrap_chat(tokenizer, user_text: str) -> str:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_text}],
            add_generation_prompt=True,
            tokenize=False,
        )


    def resolve_continuation(
        tokenizer,
        prompt: str,
        answer: str,
    ) -> tuple[list[int], list[int], str]:
        """Prompt-aware gold token ids (joint encode; try '' then ' ' separator)."""

        def enc(text: str) -> list[int]:
            return tokenizer.encode(text, add_special_tokens=False)

        prompt_ids = enc(prompt)
        answer = str(answer)
        if not answer:
            return prompt_ids, [], "EMPTY"
        candidates: list[tuple[str, list[int], int]] = []
        for sep in ("", " "):
            joint = enc(prompt + sep + answer)
            if len(joint) <= len(prompt_ids):
                continue
            if joint[: len(prompt_ids)] != prompt_ids:
                continue
            rest = joint[len(prompt_ids) :]
            candidates.append((sep, rest, len(joint)))
        if not candidates:
            bare = enc(answer)
            return prompt_ids, bare, "FALLBACK"
        candidates.sort(key=lambda c: c[2])
        sep, rest, _ = candidates[0]
        return prompt_ids, rest, repr(sep)


    @torch.inference_mode()
    def teacher_forced_metrics(
        model,
        tokenizer,
        device,
        user_text: str,
        gold_text: str,
    ) -> dict[str, Any]:
        prompt = wrap_chat(tokenizer, user_text)
        prompt_ids, gold_ids, sep_note = resolve_continuation(tokenizer, prompt, gold_text)
        n_prompt = len(prompt_ids)
        n_gold = len(gold_ids)
        if n_gold == 0:
            return {
                "n_gold_tokens": 0,
                "sum_logprob": float("nan"),
                "mean_logprob": float("nan"),
                "gold_first_token_rank": -1,
                "gold_first_token_logprob": float("nan"),
                "prompt_n_tokens": n_prompt,
                "sep_note": sep_note,
            }
        if DRY_RUN or model is None:
            return {
                "n_gold_tokens": n_gold,
                "sum_logprob": 0.0,
                "mean_logprob": 0.0,
                "gold_first_token_rank": 1,
                "gold_first_token_logprob": 0.0,
                "prompt_n_tokens": n_prompt,
                "sep_note": "DRY_RUN",
            }

        input_ids = torch.tensor([prompt_ids + gold_ids], dtype=torch.long, device=device)
        out = model(input_ids=input_ids, use_cache=False)
        # logits[t] predicts token t+1
        logits = out.logits[0]  # [seq, vocab]
        # gold token at absolute index n_prompt + i is predicted by position n_prompt + i - 1
        gold_logits = logits[n_prompt - 1 : n_prompt + n_gold - 1]
        log_probs = F.log_softmax(gold_logits.float(), dim=-1)
        gold_t = torch.tensor(gold_ids, device=device, dtype=torch.long)
        tok_lp = log_probs.gather(1, gold_t.unsqueeze(1)).squeeze(1)
        sum_lp = float(tok_lp.sum().item())
        mean_lp = sum_lp / n_gold

        first_logits = gold_logits[0].float()
        first_tid = int(gold_ids[0])
        first_lp = float(F.log_softmax(first_logits, dim=-1)[first_tid].item())
        rank = int((first_logits > first_logits[first_tid]).sum().item()) + 1

        del out, logits, input_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "n_gold_tokens": n_gold,
            "sum_logprob": round(sum_lp, 6),
            "mean_logprob": round(mean_lp, 6),
            "gold_first_token_rank": rank,
            "gold_first_token_logprob": round(first_lp, 6),
            "prompt_n_tokens": n_prompt,
            "sep_note": sep_note,
        }


    def load_model(model_id: str, quant: str):
        assert torch.cuda.is_available() or DRY_RUN, "GPU required (Colab T4) unless DRY_RUN."
        tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True)
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        if DRY_RUN:
            print(f"[model] DRY_RUN skip load: {model_id} ({quant})")
            return tok, None, torch.device("cpu")

        common = dict(
            device_map="auto",
            token=HF_TOKEN or True,
            attn_implementation="sdpa",  # T4: no FlashAttention-2
            torch_dtype=torch.float16,  # T4: fp16 only, no bf16
        )
        if quant == "fp16":
            mdl = AutoModelForCausalLM.from_pretrained(model_id, **common)
            label = "fp16 unquantized + sdpa"
        else:
            raise ValueError(f"O14b allows fp16 only, got {quant}")
        mdl.eval()
        device = next(mdl.parameters()).device
        print(f"[model] {model_id}  {label}  device={device}")
        return tok, mdl, device


    def unload(mdl):
        if mdl is None:
            return
        del mdl
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    @torch.inference_mode()
    def greedy_generate(model, tokenizer, device, user_text: str) -> str:
        if DRY_RUN or model is None:
            return "pick-up a\nstack a b"
        prompt = wrap_chat(tokenizer, user_text)
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        out = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        gen = out[0, input_ids.shape[1] :]
        text = tokenizer.decode(gen, skip_special_tokens=True)
        del out, input_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return text


    print("[o5] teacher-forced primitives ready")


## Score 360 rows × 2 models (resume-safe)

For each row: Probe-1 BW prompt → teacher-forced `mean_logprob` of `correct_answer`,
then greedy generation + `verify_answer(..., family="blocksworld")`.


In [ ]:
if not (RUN_K6_O14B):
    print('[skip] K6_O14B  (RUN_K6_O14B=False)')
else:
    MODEL_SHORT = {
        "Qwen/Qwen2.5-1.5B-Instruct": "Qwen1.5B",
        "Qwen/Qwen2.5-3B-Instruct": "Qwen3B",
    }


    def _done_keys(path: Path) -> set[tuple[str, str, str]]:
        if not path.is_file():
            return set()
        df = pd.read_csv(path, dtype=str).fillna("")
        return set(zip(df["model"], df["pair_id"], df["arm"]))


    def append_rows(path: Path, rows: list[dict[str, Any]]) -> None:
        if not rows:
            return
        new = pd.DataFrame(rows)
        for c in COLUMNS:
            if c not in new.columns:
                new[c] = ""
        new = new[COLUMNS]
        if path.is_file():
            old = pd.read_csv(path, dtype=str).fillna("")
            for c in COLUMNS:
                if c not in old.columns:
                    old[c] = ""
            old = old[COLUMNS]
            out = pd.concat([old, new], ignore_index=True)
        else:
            out = new
        out.to_csv(path, index=False)


    done = _done_keys(O14B_CSV) if RESUME else set()
    print(f"[resume] {len(done)} rows already scored; RESUME={RESUME}")

    for model_id, quant in MODELS:
        short = MODEL_SHORT[model_id]
        pending = [
            r
            for r in bank_df.to_dict(orient="records")
            if (model_id, str(r["pair_id"]), str(r["arm"])) not in done
        ]
        print(f"\n=== {model_id}  pending={len(pending)}/{len(bank_df)} ===")
        if not pending and not DRY_RUN:
            continue
        tok, mdl, device = load_model(model_id, quant)
        buf: list[dict[str, Any]] = []
        for r in tqdm(pending, desc=short):
            user = build_prompt(str(r["problem_text"]), "BW")
            gold = str(r["correct_answer"])
            tf = teacher_forced_metrics(mdl, tok, device, user, gold)
            greedy = greedy_generate(mdl, tok, device, user)
            try:
                ok = bool(
                    verify_answer(
                        str(r["problem_id"]),
                        greedy,
                        gold,
                        "blocksworld",
                        problem_text=str(r["problem_text"]),
                    )
                )
            except Exception as exc:  # noqa: BLE001
                ok = False
                greedy = f"VERIFY_ERROR: {exc}\n{greedy}"
            buf.append(
                {
                    "pair_id": r["pair_id"],
                    "arm": r["arm"],
                    "problem_id": r["problem_id"],
                    "model": model_id,
                    "model_short": short,
                    "num_blocks": r["num_blocks"],
                    "plan_length": r["plan_length"],
                    "n_gold_tokens": tf["n_gold_tokens"],
                    "sum_logprob": tf["sum_logprob"],
                    "mean_logprob": tf["mean_logprob"],
                    "gold_first_token_rank": tf["gold_first_token_rank"],
                    "gold_first_token_logprob": tf["gold_first_token_logprob"],
                    "prompt_n_tokens": tf["prompt_n_tokens"],
                    "sep_note": tf["sep_note"],
                    "greedy_response": greedy.replace("\n", "\\n"),
                    "greedy_correct": ok,
                    "binary_flag": "",  # filled in analysis
                }
            )
            if len(buf) >= 20:
                append_rows(O14B_CSV, buf)
                buf = []
        append_rows(O14B_CSV, buf)
        unload(mdl)
        done = _done_keys(O14B_CSV)

    print(f"\n[wrote] {O14B_CSV}")
    print(pd.read_csv(O14B_CSV).groupby(["model_short", "arm"]).size().unstack(fill_value=0).to_string())


## Analysis (pre-registered)

1. Paired within-pair **A vs B**, **A vs C**, **B vs C** on `mean_logprob`.
2. Cluster bootstrap on **`pair_id`**, B=5000, seed=42 → Δ ± 95% CI.
3. OLS: within-pair Δ ~ num_blocks + plan_length; also Spearman(Δ, moderator).
4. Greedy accuracy → `BINARY_DEGENERATE` if mean ≤ 0.05.


In [ ]:
if not (RUN_K6_O14B):
    print('[skip] K6_O14B  (RUN_K6_O14B=False)')
else:
    from probes.common.cluster_inference import bootstrap_p_two_sided, cluster_bootstrap_assoc

    raw = pd.read_csv(O14B_CSV, dtype=str).fillna("")
    raw["mean_logprob"] = pd.to_numeric(raw["mean_logprob"], errors="coerce")
    raw["num_blocks"] = pd.to_numeric(raw["num_blocks"], errors="coerce")
    raw["plan_length"] = pd.to_numeric(raw["plan_length"], errors="coerce")
    raw["greedy_correct"] = (
        raw["greedy_correct"].astype(str).str.strip().str.lower().isin({"true", "1", "yes"})
    )
    raw = raw.drop_duplicates(["model", "pair_id", "arm"], keep="last")

    ARM_A, ARM_B, ARM_C = "A_sequential", "B_scattered", "C_indexed"
    CONTRASTS = [
        ("A_minus_B", ARM_A, ARM_B),
        ("A_minus_C", ARM_A, ARM_C),
        ("B_minus_C", ARM_B, ARM_C),
    ]


    def cluster_mean_ci(
        values: np.ndarray,
        cluster_ids: np.ndarray,
        *,
        n_boot: int = N_BOOT,
        seed: int = SEED,
    ) -> dict[str, float]:
        df = pd.DataFrame({"v": values, "c": cluster_ids})
        per = df.groupby("c", sort=False)["v"].mean()
        estimate = float(per.mean()) if len(per) else float("nan")
        vals = per.to_numpy(dtype=float)
        if len(vals) == 0:
            return {
                "estimate": estimate,
                "ci_low": float("nan"),
                "ci_high": float("nan"),
                "p_clustered": float("nan"),
                "n_clusters": 0,
            }
        rng = np.random.default_rng(seed)
        boots = np.empty(n_boot, dtype=float)
        for i in range(n_boot):
            draw = rng.choice(len(vals), size=len(vals), replace=True)
            boots[i] = float(np.mean(vals[draw]))
        return {
            "estimate": estimate,
            "ci_low": float(np.percentile(boots, 2.5)),
            "ci_high": float(np.percentile(boots, 97.5)),
            "p_clustered": float(bootstrap_p_two_sided(boots)),
            "n_clusters": int(len(vals)),
        }


    def ols_two_moderators(y: np.ndarray, x1: np.ndarray, x2: np.ndarray) -> dict[str, float]:
        """y ~ 1 + x1 + x2 (num_blocks, plan_length)."""
        X = np.column_stack([np.ones(len(y)), x1, x2])
        coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        yhat = X @ coef
        ss_res = float(np.sum((y - yhat) ** 2))
        ss_tot = float(np.sum((y - y.mean()) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
        return {
            "intercept": float(coef[0]),
            "coef_num_blocks": float(coef[1]),
            "coef_plan_length": float(coef[2]),
            "r2": r2,
        }


    analysis_rows: list[dict[str, Any]] = []

    for model_id, gmod in raw.groupby("model"):
        short = MODEL_SHORT.get(str(model_id), str(model_id))
        acc = float(gmod["greedy_correct"].mean()) if len(gmod) else float("nan")
        binary_flag = "BINARY_DEGENERATE" if (acc == acc and acc <= FLOOR_ACC) else "BINARY_OK"
        analysis_rows.append(
            {
                "analysis": "greedy_accuracy",
                "contrast": "all_arms",
                "model": model_id,
                "model_short": short,
                "n_pairs": int(gmod["pair_id"].nunique()),
                "mean_arm_hi": "",
                "mean_arm_lo": "",
                "delta_mean": "",
                "ci_low": "",
                "ci_high": "",
                "p_clustered": "",
                "n_clusters": "",
                "greedy_accuracy": round(acc, 6),
                "binary_flag": binary_flag,
                "moderator": "",
                "spearman_r": "",
                "spearman_ci_low": "",
                "spearman_ci_high": "",
                "spearman_p": "",
                "ols_intercept": "",
                "ols_coef_num_blocks": "",
                "ols_coef_plan_length": "",
                "ols_r2": "",
                "interpretation_gate": binary_flag,
                "scope_caveat": PREREG["scope_caveat"],
            }
        )
        print(f"[{short}] greedy_acc={acc:.4f} → {binary_flag}")

        wide = gmod.pivot_table(
            index="pair_id", columns="arm", values="mean_logprob", aggfunc="last"
        )
        meta = (
            gmod.drop_duplicates("pair_id")
            .set_index("pair_id")[["num_blocks", "plan_length"]]
        )

        for contrast, arm_hi, arm_lo in CONTRASTS:
            if arm_hi not in wide.columns or arm_lo not in wide.columns:
                continue
            sub = wide[[arm_hi, arm_lo]].dropna().join(meta, how="inner")
            if sub.empty:
                continue
            delta = (sub[arm_hi] - sub[arm_lo]).to_numpy(dtype=float)
            clusters = sub.index.to_numpy()
            seed_c = (hash(f"o14b|{model_id}|{contrast}") % (2**31 - 1)) or SEED
            ci = cluster_mean_ci(delta, clusters, seed=seed_c)
            gate = (
                "naming_moves_likelihood"
                if (
                    ci["ci_low"] == ci["ci_low"]
                    and (ci["ci_low"] > 0 or ci["ci_high"] < 0)
                )
                else "naming_null_compatible"
            )
            analysis_rows.append(
                {
                    "analysis": "paired_mean_logprob_delta",
                    "contrast": contrast,
                    "model": model_id,
                    "model_short": short,
                    "n_pairs": int(len(sub)),
                    "mean_arm_hi": round(float(sub[arm_hi].mean()), 6),
                    "mean_arm_lo": round(float(sub[arm_lo].mean()), 6),
                    "delta_mean": round(ci["estimate"], 6),
                    "ci_low": round(ci["ci_low"], 6),
                    "ci_high": round(ci["ci_high"], 6),
                    "p_clustered": round(ci["p_clustered"], 6),
                    "n_clusters": ci["n_clusters"],
                    "greedy_accuracy": round(acc, 6),
                    "binary_flag": binary_flag,
                    "moderator": "",
                    "spearman_r": "",
                    "spearman_ci_low": "",
                    "spearman_ci_high": "",
                    "spearman_p": "",
                    "ols_intercept": "",
                    "ols_coef_num_blocks": "",
                    "ols_coef_plan_length": "",
                    "ols_r2": "",
                    "interpretation_gate": gate,
                    "scope_caveat": PREREG["scope_caveat"],
                }
            )
            print(
                f"  {contrast}: Δ={ci['estimate']:.4f} "
                f"[{ci['ci_low']:.4f}, {ci['ci_high']:.4f}] p={ci['p_clustered']:.4f} → {gate}"
            )

            # Scaling: OLS + Spearman per moderator
            x1 = sub["num_blocks"].to_numpy(dtype=float)
            x2 = sub["plan_length"].to_numpy(dtype=float)
            ols = ols_two_moderators(delta, x1, x2)
            analysis_rows.append(
                {
                    "analysis": "delta_ols_num_blocks_plan_length",
                    "contrast": contrast,
                    "model": model_id,
                    "model_short": short,
                    "n_pairs": int(len(sub)),
                    "mean_arm_hi": "",
                    "mean_arm_lo": "",
                    "delta_mean": round(float(np.mean(delta)), 6),
                    "ci_low": "",
                    "ci_high": "",
                    "p_clustered": "",
                    "n_clusters": int(len(sub)),
                    "greedy_accuracy": round(acc, 6),
                    "binary_flag": binary_flag,
                    "moderator": "num_blocks+plan_length",
                    "spearman_r": "",
                    "spearman_ci_low": "",
                    "spearman_ci_high": "",
                    "spearman_p": "",
                    "ols_intercept": round(ols["intercept"], 6),
                    "ols_coef_num_blocks": round(ols["coef_num_blocks"], 6),
                    "ols_coef_plan_length": round(ols["coef_plan_length"], 6),
                    "ols_r2": round(ols["r2"], 6) if ols["r2"] == ols["r2"] else "",
                    "interpretation_gate": "",
                    "scope_caveat": PREREG["scope_caveat"],
                }
            )
            for moderator, x in (("num_blocks", x1), ("plan_length", x2)):
                sp = cluster_bootstrap_assoc(
                    x,
                    delta,
                    clusters.tolist(),
                    kind="spearman",
                    n_boot=N_BOOT,
                    seed=(hash(f"o14b|{model_id}|{contrast}|{moderator}") % (2**31 - 1))
                    or SEED,
                )
                analysis_rows.append(
                    {
                        "analysis": "delta_scales_with",
                        "contrast": contrast,
                        "model": model_id,
                        "model_short": short,
                        "n_pairs": int(len(sub)),
                        "mean_arm_hi": "",
                        "mean_arm_lo": "",
                        "delta_mean": round(float(np.mean(delta)), 6),
                        "ci_low": "",
                        "ci_high": "",
                        "p_clustered": "",
                        "n_clusters": sp["n_clusters"],
                        "greedy_accuracy": round(acc, 6),
                        "binary_flag": binary_flag,
                        "moderator": moderator,
                        "spearman_r": round(sp["estimate"], 6)
                        if sp["estimate"] == sp["estimate"]
                        else "",
                        "spearman_ci_low": round(sp["ci_low"], 6)
                        if sp["ci_low"] == sp["ci_low"]
                        else "",
                        "spearman_ci_high": round(sp["ci_high"], 6)
                        if sp["ci_high"] == sp["ci_high"]
                        else "",
                        "spearman_p": round(sp["p_clustered"], 6)
                        if sp["p_clustered"] == sp["p_clustered"]
                        else "",
                        "ols_intercept": "",
                        "ols_coef_num_blocks": "",
                        "ols_coef_plan_length": "",
                        "ols_r2": "",
                        "interpretation_gate": "",
                        "scope_caveat": PREREG["scope_caveat"],
                    }
                )

    # Headline interpretation (primary contrast A−B)
    paired = [
        r
        for r in analysis_rows
        if r["analysis"] == "paired_mean_logprob_delta" and r["contrast"] == "A_minus_B"
    ]
    moves = sum(1 for r in paired if r["interpretation_gate"] == "naming_moves_likelihood")
    analysis_rows.append(
        {
            "analysis": "headline",
            "contrast": "A_minus_B",
            "model": "ALL",
            "model_short": "ALL",
            "n_pairs": "",
            "mean_arm_hi": "",
            "mean_arm_lo": "",
            "delta_mean": "",
            "ci_low": "",
            "ci_high": "",
            "p_clustered": "",
            "n_clusters": "",
            "greedy_accuracy": "",
            "binary_flag": "",
            "moderator": "",
            "spearman_r": "",
            "spearman_ci_low": "",
            "spearman_ci_high": "",
            "spearman_p": "",
            "ols_intercept": "",
            "ols_coef_num_blocks": "",
            "ols_coef_plan_length": "",
            "ols_r2": "",
            "interpretation_gate": (
                f"models_with_substantial_naming_effect={moves}/{len(paired)}; "
                + (
                    "PRIMARY_FINDING_naming_is_lexical"
                    if moves > 0
                    else "NULL_COMPATIBLE_L1_survives_on_open_models"
                )
            ),
            "scope_caveat": PREREG["scope_caveat"],
        }
    )

    out_a = pd.DataFrame(analysis_rows)
    out_a.to_csv(O14B_ANALYSIS, index=False)
    print(f"\n[wrote] {O14B_ANALYSIS}")
    print(out_a[out_a["analysis"].isin(["greedy_accuracy", "paired_mean_logprob_delta", "headline"])][
        ["analysis", "contrast", "model_short", "delta_mean", "ci_low", "ci_high", "p_clustered", "greedy_accuracy", "binary_flag", "interpretation_gate"]
    ].to_string(index=False))
    print("\nSCOPE CAVEAT:", PREREG["scope_caveat"])


## Copy into the repo

After the Colab run:

```text
colab_out/O14b_naming_likelihood.csv  →  results/raw/O14b_naming_likelihood.csv
colab_out/O14b_naming_analysis.csv    →  results/derived/O14b_naming_analysis.csv
```

Do **not** overwrite `O14_naming_bank.jsonl` or any API O14 shard paths.


### Download — K6_O14B only


In [ ]:
if not (RUN_K6_O14B):
    print("[skip download] K6_O14B")
else:
    from pathlib import Path as _P
    import shutil as _shutil
    _names = ['O14b_naming_likelihood.csv', 'O14b_naming_analysis.csv']
    _paths = [OUT_DIR / n for n in _names]
    _present = [p for p in _paths if p.is_file()]
    print(f"[download K6_O14B] present={len(_present)}/{len(_paths)} in {OUT_DIR}")
    for p in _paths:
        print(" ", "OK" if p.is_file() else "MISSING", p.name)
    _drive_dir = _P("/content/drive/MyDrive/rvc_colab_out")
    if _P("/content").exists() and not _P("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
        except Exception as _exc:
            print("[drive] mount skipped:", _exc)
    try:
        _drive_dir.mkdir(parents=True, exist_ok=True)
        for p in _present:
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
    except Exception as _exc:
        print("[backup] skipped:", _exc)
    try:
        from google.colab import files as _colab_files  # type: ignore
        for p in _present:
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
    except Exception as _exc:
        print("[download] skipped (not Colab or blocked):", _exc)


---
# K7 — DS-16 recognition vs recall

Memory-signature arm; distractors from `scripts/consolidate/ds16_distractors.py`.

**Flag:** `RUN_K7_DS16` · **Outputs (download separately):** `DS16_recognition_recall.csv`, `DS16_gap_correlations.csv`


<details><summary>Arm README (from standalone notebook)</summary>

# DS-16 — Recognition vs recall (Memory-Signature Suite arm)

**Colab T4 · Phase-0-adjacent · BX-03 / DS-16**

Memory science: **recall** and **recognition** dissociate. A *retrieved* answer
should remain recognizable even when it cannot be produced. A *computed* answer
should show a much smaller recognition–recall gap.

### Per item (canonical + W3)
| Channel | Definition |
|---------|------------|
| **RECALL** | Teacher-forced `mean_logprob` of the gold (O5 path) **and** greedy verify → `recall_correct` |
| **RECOGNITION** | Gold + **k=4** scripted distractors; score each option by `mean_logprob`; gold ranks first? + margin |

Distractors: deterministic module `scripts/consolidate/ds16_distractors.py`
(BW: goal-failing plan mutations; GSM: arithmetic slips; ALGO: wrong-related algorithm outputs). **Not hand-written.**

### Metric
`recognition_recall_gap = recognition_accuracy − recall_accuracy`  
aggregated per `(family, model, variant)`, and item-level for correlations.

### Convergent test (the one that matters)
Canonical→W3 **drop** in the gap vs:
1. Infini-gram `contamination_score` (item-level)
2. C1 intrusion fingerprint (item-level intrusion prevalence on W3 among paper-model C1 errors)

Three independent retrieval signatures agreeing would be the first real
**convergent validity** in this program (notable given prior convergence failures).

### Models
`Qwen/Qwen2.5-1.5B-Instruct`, `Qwen/Qwen2.5-3B-Instruct` — fp16, `sdpa`.

### Outputs
- `colab_out/DS16_recognition_recall.csv` → `results/derived/DS16_recognition_recall.csv`
- `colab_out/DS16_gap_correlations.csv` → `results/derived/DS16_gap_correlations.csv`

### Pre-registered interpretation (freeze before run)
- **Gap drop tracks contamination and/or C1 intrusion** (CI excludes 0, predicted sign: higher contamination / intrusion → larger can−W3 gap drop, i.e. recognition advantage collapses under rename for retrieval-like items) → convergent retrieval signature; report as primary DS-16 result.
- **Null-compatible** → recognition–recall does not add convergent validity with C1/Infini-gram on these open models; still report gaps descriptively.

</details>


In [ ]:
if not (RUN_K7_DS16):
    print('[skip] K7_DS16  (RUN_K7_DS16=False)')
else:
    MAX_NEW_TOKENS = dict(DS16_MAX_NEW_TOKENS)
    VARIANTS = ('canonical', 'W3')
    print('[K7] MAX_NEW_TOKENS', MAX_NEW_TOKENS, 'LIMIT_PER_FAMILY', LIMIT_PER_FAMILY, 'K_DISTRACTORS', K_DISTRACTORS)


## Pre-registration gate


In [ ]:
if not (RUN_K7_DS16):
    print('[skip] K7_DS16  (RUN_K7_DS16=False)')
else:
    PREREG = {
        "k_distractors": K_DISTRACTORS,
        "distractor_module": "scripts/consolidate/ds16_distractors.py",
        "recall": "O5 teacher-forced mean_logprob(gold) + greedy verify_answer → recall_correct",
        "recognition": "argmax mean_logprob over gold+k distractors; margin = lp_gold - max(lp_d)",
        "gap": "recognition_correct - recall_correct (item); mean gap per (family,model,variant)",
        "drop": "gap_canonical - gap_W3 (item-level, matched problem_id)",
        "primary_tests": [
            "Spearman(drop, contamination_score) cluster-bootstrap problem_id/clone_family",
            "Spearman(drop, c1_item_intrusion_rate) same inference",
        ],
        "predicted_sign": (
            "positive: high contamination/intrusion → larger drop (recognition advantage "
            "shrinks more under W3 for retrieval-like items)"
        ),
    }
    for k, v in PREREG.items():
        print(f"{k}: {v}")
    assert K_DISTRACTORS == 4 and N_BOOT == 5000
    print("[prereg] frozen")


## Item queue (canonical + W3) + distractor documentation check


In [ ]:
from __future__ import annotations
if not (RUN_K7_DS16):
    print('[skip] K7_DS16  (RUN_K7_DS16=False)')
else:

    import csv
    import gc
    import json
    from pathlib import Path
    from typing import Any

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer

    from probes.common.clones import algo_cluster_map
    from probes.contamination.verify import verify_answer, verify_gsm_answer
    from probes.contamination.verify_algo import verify_algo
    from scripts.consolidate.ds16_distractors import (
        K_DEFAULT,
        make_distractors,
        recognition_options,
    )

    assert K_DEFAULT == K_DISTRACTORS

    # Document generator in-run (also in module docstring)
    print(make_distractors.__doc__ or "(see scripts/consolidate/ds16_distractors.py)")
    print("module file:", (REPO_ROOT / "scripts/consolidate/ds16_distractors.py").resolve())

    PROBE1_TEMPLATE = (
        "Solve the following problem exactly and provide only the final answer "
        "in the required output format. Problem: {problem}. Format instruction: "
        "{family_specific_output_format}."
    )
    FAMILY_FORMAT = {
        "GSM": (
            "Write the final numerical answer on its own line as #### <number>. "
            "No other text after that tag."
        ),
        "ALGO": (
            "Follow the problem's required output format exactly "
            "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
        ),
        "BW": (
            "A numbered list of actions only. Each action must be one of the "
            "permitted operators with their arguments. No explanation."
        ),
    }

    MODELS = [
        ("Qwen/Qwen2.5-1.5B-Instruct", "fp16"),
        ("Qwen/Qwen2.5-3B-Instruct", "fp16"),
    ]
    MODEL_SHORT = {
        "Qwen/Qwen2.5-1.5B-Instruct": "Qwen1.5B",
        "Qwen/Qwen2.5-3B-Instruct": "Qwen3B",
    }

    OUT_MAIN = OUT_DIR / "DS16_recognition_recall.csv"
    OUT_CORR = OUT_DIR / "DS16_gap_correlations.csv"


    def _norm_vt(v: str) -> str:
        v = str(v).strip()
        return "canonical" if v.lower() == "canonical" else v.upper()


    def _strip_csv_quotes(text: str) -> str:
        s = str(text)
        if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
            s = s[1:-1]
        return s


    def build_prompt(problem_text: str, family: str) -> str:
        return PROBE1_TEMPLATE.format(
            problem=problem_text.strip(),
            family_specific_output_format=FAMILY_FORMAT[family],
        )


    def load_items(limit_per_family: int | None) -> list[dict[str, Any]]:
        specs = [
            ("GSM", REPO_ROOT / "data/problems/question_bank_gsm.csv"),
            ("ALGO", REPO_ROOT / "data/problems/question_bank_algo.csv"),
            ("BW", REPO_ROOT / "data/problems/question_bank_bw.csv"),
        ]
        cmap = algo_cluster_map()
        items: list[dict[str, Any]] = []
        for family, path in specs:
            df = pd.read_csv(path, dtype=str).fillna("")
            df["problem_id"] = df["problem_id"].astype(str).str.strip()
            df["variant_type"] = df["variant_type"].map(_norm_vt)
            df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
            df["correct_answer"] = df["correct_answer"].map(_strip_csv_quotes)
            # family-native IDs only (BW bank can mix)
            if family == "BW":
                df = df[df["problem_id"].str.startswith(("BW_", "MBW_"))]
            elif family == "ALGO":
                df = df[df["problem_id"].str.startswith(("CC_", "SP_", "WIS_"))]
            elif family == "GSM":
                df = df[df["problem_id"].str.startswith("GSM_")]
            sub = df[df["variant_type"].isin(VARIANTS)].copy()
            ids = sorted(sub["problem_id"].unique())
            if limit_per_family is not None:
                ids = ids[: int(limit_per_family)]
            sub = sub[sub["problem_id"].isin(ids)]
            for _, row in sub.iterrows():
                pid = str(row["problem_id"])
                items.append(
                    {
                        "family": family,
                        "problem_id": pid,
                        "variant": str(row["variant_type"]),
                        "problem_text": str(row["problem_text"]),
                        "gold": str(row["correct_answer"]),
                        "problem_subtype": str(row.get("problem_subtype", "") or ""),
                        "difficulty_params": str(row.get("difficulty_params", "") or ""),
                        "clone_family": (
                            cmap.get(pid, f"SINGLETON_{pid}")
                            if family == "ALGO"
                            else f"SINGLETON_{pid}"
                        ),
                    }
                )
        return items


    ITEMS = load_items(LIMIT_PER_FAMILY)
    print(f"[queue] {len(ITEMS)} items (LIMIT_PER_FAMILY={LIMIT_PER_FAMILY})")
    print(pd.DataFrame(ITEMS).groupby(["family", "variant"]).size().unstack(fill_value=0).to_string())

    # Smoke: distractors for one item per family
    for fam in ("GSM", "ALGO", "BW"):
        it = next(x for x in ITEMS if x["family"] == fam and x["variant"] == "canonical")
        opts = recognition_options(
            fam, it["problem_id"], it["variant"], it["gold"], it["problem_text"], k=K_DISTRACTORS
        )
        assert sum(o["is_gold"] for o in opts) == 1 and len(opts) == 1 + K_DISTRACTORS
        print(f"[distractors {fam}/{it['problem_id']}] {[o['option_id'] for o in opts]}")


## O5 teacher-forcing + greedy (verbatim O5 path)


In [ ]:
if not (RUN_K7_DS16):
    print('[skip] K7_DS16  (RUN_K7_DS16=False)')
else:
    def wrap_chat(tokenizer, user_text: str) -> str:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_text}],
            add_generation_prompt=True,
            tokenize=False,
        )


    def resolve_continuation(
        tokenizer,
        prompt: str,
        answer: str,
    ) -> tuple[list[int], list[int], str]:
        def enc(text: str) -> list[int]:
            return tokenizer.encode(text, add_special_tokens=False)

        prompt_ids = enc(prompt)
        answer = str(answer)
        if not answer:
            return prompt_ids, [], "EMPTY"
        candidates: list[tuple[str, list[int], int]] = []
        for sep in ("", " "):
            joint = enc(prompt + sep + answer)
            if len(joint) <= len(prompt_ids):
                continue
            if joint[: len(prompt_ids)] != prompt_ids:
                continue
            rest = joint[len(prompt_ids) :]
            candidates.append((sep, rest, len(joint)))
        if not candidates:
            bare = enc(answer)
            return prompt_ids, bare, "FALLBACK"
        candidates.sort(key=lambda c: c[2])
        sep, rest, _ = candidates[0]
        return prompt_ids, rest, repr(sep)


    @torch.inference_mode()
    def teacher_forced_metrics(
        model,
        tokenizer,
        device,
        user_text: str,
        gold_text: str,
    ) -> dict[str, Any]:
        prompt = wrap_chat(tokenizer, user_text)
        prompt_ids, gold_ids, sep_note = resolve_continuation(tokenizer, prompt, gold_text)
        n_prompt = len(prompt_ids)
        n_gold = len(gold_ids)
        if n_gold == 0:
            return {
                "n_gold_tokens": 0,
                "sum_logprob": float("nan"),
                "mean_logprob": float("nan"),
                "gold_first_token_rank": -1,
                "gold_first_token_logprob": float("nan"),
                "prompt_n_tokens": n_prompt,
                "sep_note": sep_note,
            }
        if DRY_RUN or model is None:
            return {
                "n_gold_tokens": n_gold,
                "sum_logprob": 0.0,
                "mean_logprob": float(hash(gold_text) % 1000) / -1000.0,
                "gold_first_token_rank": 1,
                "gold_first_token_logprob": 0.0,
                "prompt_n_tokens": n_prompt,
                "sep_note": "DRY_RUN",
            }

        input_ids = torch.tensor([prompt_ids + gold_ids], dtype=torch.long, device=device)
        out = model(input_ids=input_ids, use_cache=False)
        logits = out.logits[0]
        gold_logits = logits[n_prompt - 1 : n_prompt + n_gold - 1]
        log_probs = F.log_softmax(gold_logits.float(), dim=-1)
        gold_t = torch.tensor(gold_ids, device=device, dtype=torch.long)
        tok_lp = log_probs.gather(1, gold_t.unsqueeze(1)).squeeze(1)
        sum_lp = float(tok_lp.sum().item())
        mean_lp = sum_lp / n_gold
        first_logits = gold_logits[0].float()
        first_tid = int(gold_ids[0])
        first_lp = float(F.log_softmax(first_logits, dim=-1)[first_tid].item())
        rank = int((first_logits > first_logits[first_tid]).sum().item()) + 1
        del out, logits, input_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "n_gold_tokens": n_gold,
            "sum_logprob": round(sum_lp, 6),
            "mean_logprob": round(mean_lp, 6),
            "gold_first_token_rank": rank,
            "gold_first_token_logprob": round(first_lp, 6),
            "prompt_n_tokens": n_prompt,
            "sep_note": sep_note,
        }


    def load_model(model_id: str, quant: str):
        assert torch.cuda.is_available() or DRY_RUN, "GPU required unless DRY_RUN"
        tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True)
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        if DRY_RUN:
            print(f"[model] DRY_RUN {model_id}")
            return tok, None, torch.device("cpu")
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            token=HF_TOKEN or True,
            attn_implementation="sdpa",
            torch_dtype=torch.float16,
        )
        mdl.eval()
        device = next(mdl.parameters()).device
        print(f"[model] {model_id} fp16+sdpa device={device}")
        return tok, mdl, device


    def unload(mdl):
        if mdl is None:
            return
        del mdl
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    @torch.inference_mode()
    def greedy_generate(model, tokenizer, device, user_text: str, family: str) -> str:
        if DRY_RUN or model is None:
            return "DRY_RUN_ANSWER"
        prompt = wrap_chat(tokenizer, user_text)
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        out = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_NEW_TOKENS[family],
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        text = tokenizer.decode(out[0, input_ids.shape[1] :], skip_special_tokens=True)
        del out, input_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return text


    def verify_correct(item: dict[str, Any], response: str) -> bool:
        family = item["family"]
        problem_id = item["problem_id"]
        gold = item["gold"]
        problem_text = item["problem_text"]
        try:
            if family == "GSM":
                return bool(verify_gsm_answer(response, gold))
            if family == "ALGO":
                ok, _reason, _meta = verify_algo(
                    problem_id,
                    response,
                    gold,
                    item.get("problem_subtype", ""),
                    item["variant"],
                    item.get("difficulty_params", ""),
                    problem_text=problem_text,
                )
                return bool(ok)
            vf = (
                "mystery_blocksworld"
                if item.get("problem_subtype") == "mystery_blocksworld"
                or str(problem_id).startswith("MBW_")
                else "blocksworld"
            )
            return bool(
                verify_answer(
                    problem_id,
                    response,
                    gold,
                    vf,
                    problem_text=problem_text,
                )
            )
        except Exception:
            return False


    print("[o5] primitives ready")


## Score recognition + recall (log every per-option score)

Resume key: `(model, family, problem_id, variant)`.


In [ ]:
if not (RUN_K7_DS16):
    print('[skip] K7_DS16  (RUN_K7_DS16=False)')
else:
    ROW_COLUMNS = [
        "family", "problem_id", "variant", "model", "model_short", "clone_family",
        "recall_mean_logprob", "recall_sum_logprob", "recall_n_gold_tokens",
        "recall_correct", "greedy_response",
        "recognition_correct", "recognition_rank_of_gold", "recognition_margin",
        "n_options", "option_id_order",
        # per-option scores (JSON list aligned with option_id_order)
        "option_mean_logprobs", "option_is_gold_flags", "option_texts",
    ]


    def _done_keys(path: Path) -> set[tuple[str, str, str, str]]:
        if not path.is_file():
            return set()
        df = pd.read_csv(path, dtype=str).fillna("")
        return set(zip(df["model"], df["family"], df["problem_id"], df["variant"]))


    def append_rows(path: Path, rows: list[dict[str, Any]]) -> None:
        if not rows:
            return
        new = pd.DataFrame(rows)
        for c in ROW_COLUMNS:
            if c not in new.columns:
                new[c] = ""
        new = new[ROW_COLUMNS]
        if path.is_file():
            old = pd.read_csv(path, dtype=str).fillna("")
            for c in ROW_COLUMNS:
                if c not in old.columns:
                    old[c] = ""
            out = pd.concat([old[ROW_COLUMNS], new], ignore_index=True)
        else:
            out = new
        out.to_csv(path, index=False)


    done = _done_keys(OUT_MAIN) if RESUME else set()
    print(f"[resume] done={len(done)}")

    for model_id, quant in MODELS:
        short = MODEL_SHORT[model_id]
        pending = [
            it for it in ITEMS
            if (model_id, it["family"], it["problem_id"], it["variant"]) not in done
        ]
        print(f"\n=== {model_id} pending={len(pending)}/{len(ITEMS)} ===")
        if not pending:
            continue
        tok, mdl, device = load_model(model_id, quant)
        buf: list[dict[str, Any]] = []
        for it in tqdm(pending, desc=short):
            fam = it["family"]
            user = build_prompt(it["problem_text"], fam)
            gold = it["gold"]
            # Recall TF
            tf = teacher_forced_metrics(mdl, tok, device, user, gold)
            greedy = greedy_generate(mdl, tok, device, user, fam)
            recall_ok = verify_correct(it, greedy)
            # Recognition options
            opts = recognition_options(
                fam, it["problem_id"], it["variant"], gold, it["problem_text"], k=K_DISTRACTORS
            )
            scores: list[float] = []
            for o in opts:
                # DRY_RUN: perturb so gold is not always rank-1
                if DRY_RUN:
                    base = float(hash(o["text"]) % 1000) / -1000.0
                    if o["is_gold"]:
                        base += 0.02
                    scores.append(base)
                else:
                    m = teacher_forced_metrics(mdl, tok, device, user, o["text"])
                    scores.append(float(m["mean_logprob"]))
            gold_idx = next(i for i, o in enumerate(opts) if o["is_gold"])
            best = int(np.nanargmax(np.asarray(scores, dtype=float)))
            recog_ok = best == gold_idx
            gold_lp = scores[gold_idx]
            other = [s for i, s in enumerate(scores) if i != gold_idx]
            margin = float(gold_lp - max(other)) if other else float("nan")
            rank = int(1 + sum(1 for s in scores if s > gold_lp + 1e-15))
            buf.append(
                {
                    "family": fam,
                    "problem_id": it["problem_id"],
                    "variant": it["variant"],
                    "model": model_id,
                    "model_short": short,
                    "clone_family": it["clone_family"],
                    "recall_mean_logprob": tf["mean_logprob"],
                    "recall_sum_logprob": tf["sum_logprob"],
                    "recall_n_gold_tokens": tf["n_gold_tokens"],
                    "recall_correct": recall_ok,
                    "greedy_response": greedy.replace("\n", "\\n")[:2000],
                    "recognition_correct": recog_ok,
                    "recognition_rank_of_gold": rank,
                    "recognition_margin": round(margin, 6),
                    "n_options": len(opts),
                    "option_id_order": json.dumps([o["option_id"] for o in opts]),
                    "option_mean_logprobs": json.dumps([round(s, 6) for s in scores]),
                    "option_is_gold_flags": json.dumps([bool(o["is_gold"]) for o in opts]),
                    "option_texts": json.dumps([o["text"] for o in opts]),
                }
            )
            if len(buf) >= 10:
                append_rows(OUT_MAIN, buf)
                buf = []
        append_rows(OUT_MAIN, buf)
        unload(mdl)
        done = _done_keys(OUT_MAIN)

    print(f"[wrote] {OUT_MAIN}")
    summary = pd.read_csv(OUT_MAIN)
    summary["recall_correct"] = summary["recall_correct"].astype(str).str.lower().isin({"true", "1"})
    summary["recognition_correct"] = summary["recognition_correct"].astype(str).str.lower().isin({"true", "1"})
    summary["gap"] = summary["recognition_correct"].astype(float) - summary["recall_correct"].astype(float)
    print(summary.groupby(["model_short", "family", "variant"])[["recall_correct", "recognition_correct", "gap"]].mean().round(3).to_string())


## Gaps + convergent correlations

Item-level `drop = gap_can − gap_W3`. Correlate with Infini-gram contamination and
C1 item intrusion rate (fraction of C1 W3 errors on that item that are INTRUSION).


In [ ]:
if not (RUN_K7_DS16):
    print('[skip] K7_DS16  (RUN_K7_DS16=False)')
else:
    from probes.common.cluster_inference import cluster_bootstrap_assoc

    raw = pd.read_csv(OUT_MAIN, dtype=str).fillna("")
    raw["recall_correct"] = raw["recall_correct"].astype(str).str.lower().isin({"true", "1", "yes"})
    raw["recognition_correct"] = raw["recognition_correct"].astype(str).str.lower().isin({"true", "1", "yes"})
    raw["gap"] = raw["recognition_correct"].astype(float) - raw["recall_correct"].astype(float)
    raw["recognition_margin"] = pd.to_numeric(raw["recognition_margin"], errors="coerce")
    raw["recall_mean_logprob"] = pd.to_numeric(raw["recall_mean_logprob"], errors="coerce")

    # --- aggregate gaps ---
    agg_rows: list[dict[str, Any]] = []
    for (model, fam, variant), g in raw.groupby(["model", "family", "variant"]):
        agg_rows.append(
            {
                "analysis": "gap_by_cell",
                "family": fam,
                "model": model,
                "model_short": MODEL_SHORT.get(model, model),
                "variant": variant,
                "n": int(len(g)),
                "recall_accuracy": round(float(g["recall_correct"].mean()), 6),
                "recognition_accuracy": round(float(g["recognition_correct"].mean()), 6),
                "recognition_recall_gap": round(float(g["gap"].mean()), 6),
                "mean_recognition_margin": round(float(g["recognition_margin"].mean()), 6),
                "spearman_rho": "",
                "ci_low": "",
                "ci_high": "",
                "p_clustered": "",
                "n_clusters": "",
                "verdict": "",
                "note": "gap = recognition_acc - recall_acc",
            }
        )

    # --- item-level drop ---
    corr_rows: list[dict[str, Any]] = list(agg_rows)

    # Infini-gram
    cont_parts = []
    for fam, name in (
        ("GSM", "GSM_P3_contamination.csv"),
        ("ALGO", "ALGO_P3_contamination.csv"),
        ("BW", "BW_P3_contamination.csv"),
    ):
        path = REPO_ROOT / "results/raw" / name
        if not path.is_file():
            print("[warn] missing", path)
            continue
        c = pd.read_csv(path)
        c["family"] = fam
        c["problem_id"] = c["problem_id"].astype(str).str.strip()
        c["contamination_score"] = pd.to_numeric(c["contamination_score"], errors="coerce")
        cont_parts.append(
            c[["family", "problem_id", "contamination_score"]].drop_duplicates(
                ["family", "problem_id"]
            )
        )
    cont = pd.concat(cont_parts, ignore_index=True) if cont_parts else pd.DataFrame()

    # C1 item intrusion rate on W3
    c1_path = REPO_ROOT / "results/derived/C1_intrusion_errors.csv"
    if c1_path.is_file():
        c1 = pd.read_csv(c1_path, dtype=str).fillna("")
        c1 = c1[c1["variant"].astype(str).str.upper().eq("W3")].copy()
        c1["is_intrusion"] = c1["error_class"].astype(str).str.upper().eq("INTRUSION")
        c1_rate = (
            c1.groupby(["family", "problem_id"], as_index=False)
            .agg(
                c1_n_errors=("is_intrusion", "size"),
                c1_n_intrusion=("is_intrusion", "sum"),
            )
        )
        c1_rate["c1_item_intrusion_rate"] = c1_rate["c1_n_intrusion"] / c1_rate["c1_n_errors"].clip(lower=1)
    else:
        c1_rate = pd.DataFrame(columns=["family", "problem_id", "c1_item_intrusion_rate"])
        print("[warn] missing C1_intrusion_errors.csv")

    for model_id, gmod in raw.groupby("model"):
        short = MODEL_SHORT.get(model_id, model_id)
        can = gmod[gmod["variant"] == "canonical"][
            ["family", "problem_id", "clone_family", "gap"]
        ].rename(columns={"gap": "gap_can"})
        w3 = gmod[gmod["variant"] == "W3"][
            ["family", "problem_id", "gap"]
        ].rename(columns={"gap": "gap_w3"})
        merged = can.merge(w3, on=["family", "problem_id"], how="inner")
        merged["drop"] = merged["gap_can"] - merged["gap_w3"]
        if len(cont):
            merged = merged.merge(cont, on=["family", "problem_id"], how="left")
        else:
            merged["contamination_score"] = np.nan
        if len(c1_rate):
            merged = merged.merge(c1_rate[["family", "problem_id", "c1_item_intrusion_rate"]], on=["family", "problem_id"], how="left")
        else:
            merged["c1_item_intrusion_rate"] = np.nan

        for fam, gf in merged.groupby("family"):
            clusters = gf["clone_family"].tolist()
            for xname, predicted in (
                ("contamination_score", "positive"),
                ("c1_item_intrusion_rate", "positive"),
            ):
                sub = gf.dropna(subset=["drop", xname])
                if len(sub) < 8:
                    corr_rows.append(
                        {
                            "analysis": "drop_vs_" + xname,
                            "family": fam,
                            "model": model_id,
                            "model_short": short,
                            "variant": "can_minus_W3",
                            "n": int(len(sub)),
                            "recall_accuracy": "",
                            "recognition_accuracy": "",
                            "recognition_recall_gap": "",
                            "mean_recognition_margin": "",
                            "spearman_rho": "",
                            "ci_low": "",
                            "ci_high": "",
                            "p_clustered": "",
                            "n_clusters": "",
                            "verdict": "insufficient_n",
                            "note": f"need ≥8 items; predicted_sign={predicted}",
                        }
                    )
                    continue
                res = cluster_bootstrap_assoc(
                    sub[xname].to_numpy(dtype=float),
                    sub["drop"].to_numpy(dtype=float),
                    sub["clone_family"].tolist(),
                    kind="spearman",
                    n_boot=N_BOOT,
                    seed=SEED,
                )
                rho = res["estimate"]
                lo, hi = res["ci_low"], res["ci_high"]
                # predicted positive: CI entirely > 0
                if lo == lo and lo > 0:
                    verdict = "convergent_positive"
                elif hi == hi and hi < 0:
                    verdict = "opposite_sign"
                else:
                    verdict = "null_compatible"
                corr_rows.append(
                    {
                        "analysis": "drop_vs_" + xname,
                        "family": fam,
                        "model": model_id,
                        "model_short": short,
                        "variant": "can_minus_W3",
                        "n": int(len(sub)),
                        "recall_accuracy": "",
                        "recognition_accuracy": "",
                        "recognition_recall_gap": round(float(sub["drop"].mean()), 6),
                        "mean_recognition_margin": "",
                        "spearman_rho": round(rho, 6) if rho == rho else "",
                        "ci_low": round(lo, 6) if lo == lo else "",
                        "ci_high": round(hi, 6) if hi == hi else "",
                        "p_clustered": round(res["p_clustered"], 6)
                        if res["p_clustered"] == res["p_clustered"]
                        else "",
                        "n_clusters": res["n_clusters"],
                        "verdict": verdict,
                        "note": (
                            f"drop=gap_can-gap_W3; cluster=clone_family; "
                            f"predicted_sign={predicted}; {PREREG['predicted_sign']}"
                        ),
                    }
                )
                print(
                    f"[{short}/{fam}] drop vs {xname}: ρ={rho:.3f} "
                    f"[{lo:.3f},{hi:.3f}] → {verdict} (n={len(sub)})"
                )

    # Headline: any convergent_positive?
    n_pos = sum(1 for r in corr_rows if r.get("verdict") == "convergent_positive")
    n_tests = sum(1 for r in corr_rows if r.get("analysis", "").startswith("drop_vs_"))
    corr_rows.append(
        {
            "analysis": "headline",
            "family": "ALL",
            "model": "ALL",
            "model_short": "ALL",
            "variant": "can_minus_W3",
            "n": n_tests,
            "recall_accuracy": "",
            "recognition_accuracy": "",
            "recognition_recall_gap": "",
            "mean_recognition_margin": "",
            "spearman_rho": "",
            "ci_low": "",
            "ci_high": "",
            "p_clustered": "",
            "n_clusters": "",
            "verdict": (
                "CONVERGENT_VALIDITY_SIGNAL"
                if n_pos > 0
                else "NULL_COMPATIBLE_no_convergent_signal"
            ),
            "note": f"n_convergent_positive_cells={n_pos}/{n_tests}",
        }
    )

    out = pd.DataFrame(corr_rows)
    out.to_csv(OUT_CORR, index=False)
    # Also copy main into derived-friendly name already OUT_MAIN
    print(f"[wrote] {OUT_CORR}")
    print(out[out["analysis"] == "headline"][["verdict", "note"]].to_string(index=False))
    print(
        out[out["analysis"] == "gap_by_cell"][
            ["model_short", "family", "variant", "recall_accuracy", "recognition_accuracy", "recognition_recall_gap"]
        ].to_string(index=False)
    )


## Copy into the repo

```text
colab_out/DS16_recognition_recall.csv   →  results/derived/DS16_recognition_recall.csv
colab_out/DS16_gap_correlations.csv     →  results/derived/DS16_gap_correlations.csv
```

Distractor generator (auditable): `scripts/consolidate/ds16_distractors.py`.


### Download — K7_DS16 only


In [ ]:
if not (RUN_K7_DS16):
    print("[skip download] K7_DS16")
else:
    from pathlib import Path as _P
    import shutil as _shutil
    _names = ['DS16_recognition_recall.csv', 'DS16_gap_correlations.csv']
    _paths = [OUT_DIR / n for n in _names]
    _present = [p for p in _paths if p.is_file()]
    print(f"[download K7_DS16] present={len(_present)}/{len(_paths)} in {OUT_DIR}")
    for p in _paths:
        print(" ", "OK" if p.is_file() else "MISSING", p.name)
    _drive_dir = _P("/content/drive/MyDrive/rvc_colab_out")
    if _P("/content").exists() and not _P("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
        except Exception as _exc:
            print("[drive] mount skipped:", _exc)
    try:
        _drive_dir.mkdir(parents=True, exist_ok=True)
        for p in _present:
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
    except Exception as _exc:
        print("[backup] skipped:", _exc)
    try:
        from google.colab import files as _colab_files  # type: ignore
        for p in _present:
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
    except Exception as _exc:
        print("[download] skipped (not Colab or blocked):", _exc)


---
# K5 — O16 open-model calibration (last)

Close the loop. With ~0 exact / 2 near-exact corpus members, surprisal ROC has almost no positives — expect an uninformative AUC; still run to finish.

**Flag:** `RUN_K5_O16` · **Outputs (download separately):** `O16_open_model_scores.csv`


<details><summary>Arm README (from standalone notebook)</summary>

# O16 Part B — Open-corpus model scores for GT calibration (Colab T4)

Part A searched **The Pile** and **Dolma** for exact/near-exact matches of every
canonical problem. Here we score those same instances with the models trained
on those corpora:

| Model | Corpus GT | dtype |
|-------|-----------|-------|
| `EleutherAI/pythia-2.8b` | Pile (`v4_piletrain_llama`) | fp16 |
| `allenai/OLMo-2-0425-1B` | Dolma (`v4_dolma-v1_7_llama`) | fp16 |

### Measures (canonical only)
1. **O15 statement surprisal:** mean per-token NLL of the problem statement; min-k% (k=20).
2. **O5 teacher-forced gold:** mean logprob / NLL of the bank gold under the Appendix-N Probe-1 prompt.

**T4:** fp16, `attn_implementation="sdpa"`.

**Output:** `colab_out/O16_open_model_scores.csv` → `results/raw/O16_open_model_scores.csv`

Then run:
```bash
python scripts/consolidate/o16_calibrate_proxies.py
```

**Paper caveat:** this GT calibration cannot be done for Claude, GPT-4o, Gemini,
o4-mini, or DeepSeek — a permanent limitation of contamination research on closed models.

</details>


## Canonical item queue


In [ ]:
from __future__ import annotations
if not (RUN_K5_O16):
    print('[skip] K5_O16  (RUN_K5_O16=False)')
else:

    import csv
    import gc
    from typing import Any

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer

    from probes.common.clones import algo_cluster_map

    PROBE1_TEMPLATE = (
        "Solve the following problem exactly and provide only the final answer "
        "in the required output format. Problem: {problem}. Format instruction: "
        "{family_specific_output_format}."
    )
    FAMILY_FORMAT = {
        "GSM": (
            "Write the final numerical answer on its own line as #### <number>. "
            "No other text after that tag."
        ),
        "ALGO": (
            "Follow the problem's required output format exactly "
            "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
        ),
        "BW": (
            "A numbered list of actions only. Each action must be one of the "
            "permitted operators with their arguments. No explanation."
        ),
    }

    MODELS = [
        ("EleutherAI/pythia-2.8b", "pile", "Pythia/Pile"),
        ("allenai/OLMo-2-0425-1B", "dolma", "OLMo/Dolma"),
    ]

    O16_CSV = OUT_DIR / "O16_open_model_scores.csv"
    OUT_COLUMNS = [
        "family", "problem_id", "variant", "model", "corpus_lineage",
        "clone_family",
        # O15
        "o15_n_tokens", "o15_mean_nll", "o15_sum_nll", "o15_residual_mean_nll",
        "o15_min_k_mean_logprob", "o15_min_k_mean_nll", "o15_n_min_k_tokens",
        # O5
        "o5_prompt_n_tokens", "o5_n_gold_tokens",
        "o5_mean_logprob", "o5_sum_logprob", "o5_mean_nll_gold",
        "o5_gold_first_token_logprob", "o5_gold_first_token_rank",
    ]


    def _norm_vt(v: str) -> str:
        v = str(v).strip()
        return "canonical" if v.lower() == "canonical" else v.upper()


    def _strip_csv_quotes(text: str) -> str:
        s = str(text)
        if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
            s = s[1:-1]
        return s


    def load_canonicals(limit: int | None) -> list[dict[str, Any]]:
        specs = [
            ("GSM", REPO_ROOT / "data/problems/question_bank_gsm.csv"),
            ("ALGO", REPO_ROOT / "data/problems/question_bank_algo.csv"),
            ("BW", REPO_ROOT / "data/problems/question_bank_bw.csv"),
        ]
        cmap = algo_cluster_map()
        items: list[dict[str, Any]] = []
        for family, path in specs:
            df = pd.read_csv(path, dtype=str).fillna("")
            df["variant_type"] = df["variant_type"].map(_norm_vt)
            can = df[df["variant_type"] == "canonical"]
            for _, row in can.iterrows():
                pid = str(row["problem_id"]).strip()
                items.append(
                    {
                        "family": family,
                        "problem_id": pid,
                        "variant": "canonical",
                        "problem_text": _strip_csv_quotes(str(row["problem_text"])).strip(),
                        "gold": _strip_csv_quotes(str(row["correct_answer"])),
                        "clone_family": (
                            cmap.get(pid, f"SINGLETON_{pid}")
                            if family == "ALGO"
                            else f"SINGLETON_{pid}"
                        ),
                    }
                )
            if limit is not None:
                fam_ids = sorted({x["problem_id"] for x in items if x["family"] == family})[:limit]
                keep = set(fam_ids)
                items = [x for x in items if x["family"] != family or x["problem_id"] in keep]
        return items


    ITEMS = load_canonicals(LIMIT)
    print(f"[queue] {len(ITEMS)} canonical items")
    print(pd.DataFrame(ITEMS).groupby("family").size().to_string())


## Scoring functions (O15 statement + O5 teacher-forced gold)


In [ ]:
if not (RUN_K5_O16):
    print('[skip] K5_O16  (RUN_K5_O16=False)')
else:
    def append_rows(path: Path, rows: list[dict]) -> None:
        write_header = not path.exists() or path.stat().st_size == 0
        with path.open("a", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=OUT_COLUMNS)
            if write_header:
                w.writeheader()
            for r in rows:
                w.writerow({c: r.get(c, "") for c in OUT_COLUMNS})


    def _done_keys(path: Path) -> set[tuple[str, str, str]]:
        if not path.exists() or path.stat().st_size == 0:
            return set()
        df = pd.read_csv(path, dtype=str).fillna("")
        return {
            (str(r.family), str(r.problem_id), str(r.model))
            for r in df.itertuples(index=False)
        }


    def wrap_chat(tokenizer, user_text: str) -> str:
        """Best-effort chat wrap; base models fall back to raw user text."""
        if getattr(tokenizer, "chat_template", None):
            try:
                return tokenizer.apply_chat_template(
                    [{"role": "user", "content": user_text}],
                    tokenize=False,
                    add_generation_prompt=True,
                )
            except Exception:
                pass
        return user_text


    def build_prompt(problem_text: str, family: str) -> str:
        return PROBE1_TEMPLATE.format(
            problem=problem_text.strip(),
            family_specific_output_format=FAMILY_FORMAT[family],
        )


    @torch.inference_mode()
    def statement_surprisal(model, tokenizer, device, text: str) -> dict[str, Any]:
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=True)
        input_ids = enc["input_ids"].to(device)
        n = int(input_ids.shape[1])
        if n < 2:
            return {
                "o15_n_tokens": n, "o15_mean_nll": float("nan"), "o15_sum_nll": float("nan"),
                "o15_min_k_mean_logprob": float("nan"), "o15_min_k_mean_nll": float("nan"),
                "o15_n_min_k_tokens": 0,
            }
        if DRY_RUN or model is None:
            return {
                "o15_n_tokens": n - 1, "o15_mean_nll": 2.5, "o15_sum_nll": 2.5 * (n - 1),
                "o15_min_k_mean_logprob": -3.0, "o15_min_k_mean_nll": 3.0,
                "o15_n_min_k_tokens": max(1, int(np.ceil((n - 1) * MIN_K_PCT / 100))),
            }
        out = model(input_ids=input_ids, use_cache=False)
        logits = out.logits[0, :-1].float()
        targets = input_ids[0, 1:]
        log_probs = F.log_softmax(logits, dim=-1)
        tok_lp = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1).cpu().numpy()
        n_scored = len(tok_lp)
        mean_nll = float((-tok_lp).mean())
        k = max(1, int(np.ceil(n_scored * MIN_K_PCT / 100.0)))
        lowest = np.sort(tok_lp)[:k]
        min_k_lp = float(lowest.mean())
        del out, logits, input_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "o15_n_tokens": n_scored,
            "o15_mean_nll": round(mean_nll, 6),
            "o15_sum_nll": round(float((-tok_lp).sum()), 6),
            "o15_min_k_mean_logprob": round(min_k_lp, 6),
            "o15_min_k_mean_nll": round(-min_k_lp, 6),
            "o15_n_min_k_tokens": k,
        }


    def resolve_continuation(tokenizer, prompt: str, answer: str):
        enc = lambda t: tokenizer(t, add_special_tokens=False)["input_ids"]
        prompt_ids = enc(prompt)
        candidates = []
        for sep in ("", "\n", " "):
            joint = enc(prompt + sep + answer)
            if joint[: len(prompt_ids)] != prompt_ids:
                continue
            rest = joint[len(prompt_ids):]
            candidates.append((sep, rest, len(joint)))
        if not candidates:
            return prompt_ids, enc(answer)
        candidates.sort(key=lambda c: c[2])
        return prompt_ids, candidates[0][1]


    @torch.inference_mode()
    def teacher_forced_gold(model, tokenizer, device, user_text: str, gold: str) -> dict[str, Any]:
        prompt = wrap_chat(tokenizer, user_text)
        prompt_ids, gold_ids = resolve_continuation(tokenizer, prompt, gold)
        n_prompt, n_gold = len(prompt_ids), len(gold_ids)
        if n_gold == 0:
            return {
                "o5_prompt_n_tokens": n_prompt, "o5_n_gold_tokens": 0,
                "o5_mean_logprob": float("nan"), "o5_sum_logprob": float("nan"),
                "o5_mean_nll_gold": float("nan"),
                "o5_gold_first_token_logprob": float("nan"), "o5_gold_first_token_rank": -1,
            }
        if DRY_RUN or model is None:
            return {
                "o5_prompt_n_tokens": n_prompt, "o5_n_gold_tokens": n_gold,
                "o5_mean_logprob": -1.5, "o5_sum_logprob": -1.5 * n_gold,
                "o5_mean_nll_gold": 1.5,
                "o5_gold_first_token_logprob": -1.2, "o5_gold_first_token_rank": 3,
            }
        input_ids = torch.tensor([prompt_ids + gold_ids], dtype=torch.long, device=device)
        out = model(input_ids=input_ids, use_cache=False)
        gold_logits = out.logits[0, n_prompt - 1 : n_prompt + n_gold - 1].float()
        log_probs = F.log_softmax(gold_logits, dim=-1)
        gold_t = torch.tensor(gold_ids, device=device, dtype=torch.long)
        tok_lp = log_probs.gather(1, gold_t.unsqueeze(1)).squeeze(1)
        sum_lp = float(tok_lp.sum().item())
        mean_lp = sum_lp / n_gold
        first_logits = gold_logits[0]
        first_tid = int(gold_ids[0])
        first_lp = float(F.log_softmax(first_logits, dim=-1)[first_tid].item())
        rank = int((first_logits > first_logits[first_tid]).sum().item()) + 1
        del out, input_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "o5_prompt_n_tokens": n_prompt,
            "o5_n_gold_tokens": n_gold,
            "o5_mean_logprob": round(mean_lp, 6),
            "o5_sum_logprob": round(sum_lp, 6),
            "o5_mean_nll_gold": round(-mean_lp, 6),
            "o5_gold_first_token_logprob": round(first_lp, 6),
            "o5_gold_first_token_rank": rank,
        }


    def load_model(model_id: str):
        assert torch.cuda.is_available() or DRY_RUN, "GPU required unless DRY_RUN"
        tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True, trust_remote_code=True)
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        if DRY_RUN:
            print(f"[model] DRY_RUN skip {model_id}")
            return tok, None, torch.device("cpu")
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            token=HF_TOKEN or True,
            attn_implementation="sdpa",
            torch_dtype=torch.float16,
            trust_remote_code=True,
        )
        mdl.eval()
        device = next(mdl.parameters()).device
        print(f"[model] {model_id} fp16 device={device}")
        return tok, mdl, device


    def unload(mdl):
        if mdl is None:
            return
        del mdl
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## Run


In [ ]:
if not (RUN_K5_O16):
    print('[skip] K5_O16  (RUN_K5_O16=False)')
else:
    done = _done_keys(O16_CSV) if RESUME else set()
    print(f"[resume] {len(done)} rows in {O16_CSV}")

    for model_id, corpus, lineage in MODELS:
        tok, mdl, device = load_model(model_id)
        buf: list[dict] = []
        todo = [it for it in ITEMS if (it["family"], it["problem_id"], model_id) not in done]
        print(f"[run] {model_id}: {len(todo)} remaining")
        for it in tqdm(todo, desc=lineage):
            o15 = statement_surprisal(mdl, tok, device, it["problem_text"])
            user = build_prompt(it["problem_text"], it["family"])
            o5 = teacher_forced_gold(mdl, tok, device, user, it["gold"])
            row = {
                "family": it["family"],
                "problem_id": it["problem_id"],
                "variant": "canonical",
                "model": model_id,
                "corpus_lineage": corpus,
                "clone_family": it["clone_family"],
                **o15,
                **o5,
            }
            buf.append(row)
            if len(buf) >= 16:
                append_rows(O16_CSV, buf)
                buf = []
        if buf:
            append_rows(O16_CSV, buf)
        unload(mdl)
        del tok
        gc.collect()

    # Length residual within model (O15)
    if O16_CSV.exists():
        df = pd.read_csv(O16_CSV)
        parts = []
        for _, g in df.groupby("model"):
            sub = g.copy()
            x = pd.to_numeric(sub["o15_n_tokens"], errors="coerce").to_numpy(float)
            y = pd.to_numeric(sub["o15_mean_nll"], errors="coerce").to_numpy(float)
            m = np.isfinite(x) & np.isfinite(y)
            resid = np.full(len(sub), np.nan)
            if m.sum() >= 3 and np.unique(x[m]).size >= 2:
                b, a = np.polyfit(x[m], y[m], 1)
                resid[m] = y[m] - (a + b * x[m])
            sub["o15_residual_mean_nll"] = resid
            parts.append(sub)
        out = pd.concat(parts, ignore_index=True)
        # Ensure column exists in schema for downstream
        if "o15_residual_mean_nll" not in OUT_COLUMNS:
            pass
        out.to_csv(O16_CSV, index=False)
        print(out.groupby(["model", "family"]).size().unstack(fill_value=0).to_string())
        print(f"[done] {O16_CSV} ({len(out)} rows)")


## Download → `results/raw/O16_open_model_scores.csv`


### Download — K5_O16 only


In [ ]:
if not (RUN_K5_O16):
    print("[skip download] K5_O16")
else:
    from pathlib import Path as _P
    import shutil as _shutil
    _names = ['O16_open_model_scores.csv']
    _paths = [OUT_DIR / n for n in _names]
    _present = [p for p in _paths if p.is_file()]
    print(f"[download K5_O16] present={len(_present)}/{len(_paths)} in {OUT_DIR}")
    for p in _paths:
        print(" ", "OK" if p.is_file() else "MISSING", p.name)
    _drive_dir = _P("/content/drive/MyDrive/rvc_colab_out")
    if _P("/content").exists() and not _P("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
        except Exception as _exc:
            print("[drive] mount skipped:", _exc)
    try:
        _drive_dir.mkdir(parents=True, exist_ok=True)
        for p in _present:
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
    except Exception as _exc:
        print("[backup] skipped:", _exc)
    try:
        from google.colab import files as _colab_files  # type: ignore
        for p in _present:
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
    except Exception as _exc:
        print("[download] skipped (not Colab or blocked):", _exc)


## Final — download every present suite artifact

Runs regardless of individual `RUN_K*` (downloads whatever files exist in `colab_out/`).


In [ ]:
from pathlib import Path as _P
import shutil as _shutil

_ALL = [
    "O5_teacher_forced_likelihood.csv",
    "O6_quantization_sensitivity.csv",
    "O6_quantization_sensitivity_items.csv",
    "O6_quantization_sensitivity_summary.txt",
    "O6_subsample_manifest.json",
    "O15_surprisal_contamination.csv",
    "O8_mech_behavior_link.csv",
    "O8_layer_profile.csv",
    "O8_w3_binary_scores.csv",
    "O8_framing.txt",
    "O14b_naming_likelihood.csv",
    "O14b_naming_analysis.csv",
    "DS16_recognition_recall.csv",
    "DS16_gap_correlations.csv",
    "O16_open_model_scores.csv",
]
_paths = [OUT_DIR / n for n in _ALL]
_present = [p for p in _paths if p.is_file()]
print(f"[suite final] {len(_present)}/{len(_ALL)} files in {OUT_DIR}")
for p in _paths:
    print(" ", "OK" if p.is_file() else "·", p.name)

_drive_dir = _P("/content/drive/MyDrive/rvc_colab_out")
if _P("/content").exists() and not _P("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as _exc:
        print("[drive] mount skipped:", _exc)
try:
    _drive_dir.mkdir(parents=True, exist_ok=True)
    for p in _present:
        _shutil.copy2(p, _drive_dir / p.name)
        print(f"[backup] {p.name}")
except Exception as _exc:
    print("[backup] skipped:", _exc)
try:
    from google.colab import files as _colab_files  # type: ignore
    for p in _present:
        _colab_files.download(str(p))
except Exception as _exc:
    print("[download] skipped:", _exc)
